# Formula 1 Podium Prediction Companion - Milestone 1

Predict `positionOrder <= 3` for one recorded driver-race entry using information
available after qualifying/final grid and before the race. Retain 1950-2024.

| Section | Contents |
|---|---|
| 1 | Raw audit, cleaning, outcome-independent pre-race inputs and regression checks |
| 2 | Three multi-table engineering answers and uncertainty/scope checks |
| 3 | Earlier-race form/standings, chronological split and historical preparation |
| 4 | Logistic regression, random forest and shallow neural-network variants |
| 5 | Retrained feature ablations, preprocessing/fill and row-policy experiments for all families |
| 6 | Global permutation importance and SHAP plus local SHAP for every family |
| 7 | Validated complete/partial inference and boundary regression checks |
| 8 | Same-run exports, dependency/data manifest and fitted model artifacts |

**Confirmed verbal TA guidance, relayed by the team on 17 October 2026:** retain
earlier years; grid can replace qualifying predictors, including explicit grid-zero
flags; fill after splitting using strictly earlier races; repeat Parts 5 and 6 for
all three families. Part 5 must not scramble records: feature trials add/remove
groups and retrain. Required permutation importance remains in Part 6. Row or
feature removal needs justification and a visible controlled comparison.

**Evaluation limitation:** 2022-2024 was already viewed in earlier project revisions.
Current choices use validation only. This is a disclosed revised later-era evaluation,
not a newly untouched holdout. TA acceptance or genuinely new unexamined data is
needed to settle the strict unseen-test requirement.


## 0. Setup

Run All in a fresh kernel. Required libraries are NumPy, pandas, Matplotlib, Seaborn, missingno, scikit-learn, TensorFlow/Keras, SHAP and IPython. Kaggle normally provides them. No package is installed by the notebook. All random choices use seed 42. Actual versions and generated tables are exported at the end.

In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
from pathlib import Path
import json
import hashlib
PROJECT_REVISION = 'ms1-audit-fixes-2026-10-18-r2'
from IPython.display import display, Markdown
import missingno as msno
sns.set_theme(style='whitegrid', palette=['#287D8E', '#B95D4C', '#8796A5'])


Error: No connection selected.

### 0.2 Data path

On Kaggle the dataset is attached with **Add Input** and appears under `/kaggle/input/`. Kaggle has used two folder layouts (the labs use the newer `/kaggle/input/datasets/...` one), so both are checked. When running locally, the CSVs are expected in a folder called `archive` next to the notebook.

In [19]:
DATA_CANDIDATES = [
    os.environ.get('F1_DATA_DIR'),
    '/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020',
    '/kaggle/input/formula-1-world-championship-1950-2020',
    'archive', 'F1 dataset', '../archive', '../F1 dataset',
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p and Path(p, 'results.csv').is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Attach the F1 Kaggle dataset, set F1_DATA_DIR, or put the 14 CSVs in archive beside this notebook.')
OUTPUT_PATH = Path(os.environ['F1_OUTPUT_DIR']) if os.environ.get('F1_OUTPUT_DIR') else (
    Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('artifacts'))
FIG_PATH = OUTPUT_PATH / 'figures'
FIG_PATH.mkdir(parents=True, exist_ok=True)
# Windows sandboxes may deny delete-on-close files in the system temp folder.
# Keep Keras serialization scratch files inside the writable output directory.
if os.name == 'nt':
    import tempfile
    task_temp_path = (OUTPUT_PATH / '.temporary').resolve()
    task_temp_path.mkdir(exist_ok=True)
    tempfile.tempdir = str(task_temp_path)
print('Revision:', PROJECT_REVISION, '| Reading:', DATA_PATH)


Error: No connection selected.

## 1. Data inspection, auditing and cleaning

### 1.1 Load the data

The dataset is **relational**: 14 linked CSV tables.

We load every table **as plain text** (`dtype=str`) and tell pandas **not** to guess missing values (`keep_default_na=False`). This gives us an untouched copy of exactly what is in the files, which we need in order to find the placeholder values used for "missing".

* `raw` = the untouched copy. It is **never modified**.
* `data` = the working copy that we clean step by step.

In [20]:
TABLE_NAMES = ['circuits', 'constructors', 'constructor_results', 'constructor_standings',
               'drivers', 'driver_standings', 'races', 'results', 'qualifying',
               'lap_times', 'pit_stops', 'sprint_results', 'seasons', 'status']

raw = {}
for name in TABLE_NAMES:
    raw[name] = pd.read_csv(f'{DATA_PATH}/{name}.csv', dtype=str, keep_default_na=False)

source_csv_hashes = {name + '.csv': hashlib.sha256(Path(DATA_PATH, name + '.csv').read_bytes()).hexdigest() for name in TABLE_NAMES}

# Working copies: all cleaning happens on these, the raw files stay immutable
data = {name: raw[name].copy() for name in TABLE_NAMES}

for name in TABLE_NAMES:
    print(f'{name:<22} {raw[name].shape[0]:>7} rows  x {raw[name].shape[1]:>2} columns')


Error: No connection selected.

### 1.2 Inspect the data (before cleaning)

#### 1.2.1 What does a normal `read_csv` see?

We load `results.csv` with the default parser and report its shape, column types, and missing-value counts. These checks summarize the full table without displaying selected race records.

In [21]:
results_naive = pd.read_csv(f'{DATA_PATH}/results.csv')
print('Results shape:', results_naive.shape)
print('Column types:')
print(results_naive.dtypes.to_string())

Error: No connection selected.

In [22]:
results_naive.info()

Error: No connection selected.

In [23]:
results_naive.isnull().sum()

Error: No connection selected.

`isnull().sum()` reports **zero** missing values. Instead of reading `info()` by eye to find suspicious columns, we test every text column programmatically. If most of its values can be converted to a number, the column is meant to be numeric, and the values that fail to convert are the suspects.

In [24]:
suspect_rows = []
for col in results_naive.columns:
    if results_naive[col].dtype == 'object' or str(results_naive[col].dtype) == 'str':
        as_number = pd.to_numeric(results_naive[col], errors='coerce')   # non-numbers -> NaN
        share_numeric = as_number.notna().mean()
        if share_numeric > 0.1:                                          # mostly a numeric column
            failing = results_naive[col][as_number.isna()]
            suspect_rows.append({'column': col,
                                 '% values that are numbers': round(100 * share_numeric, 1),
                                 'most common non-number': failing.value_counts().index[0],
                                 'its count': failing.value_counts().iloc[0]})

pd.DataFrame(suspect_rows)

Error: No connection selected.

**Observation:** every numeric column that pandas left as text fails for the **same reason**: the literal text **`\N`** (a *sentinel*). The only exception is `positionText`, whose most common non-number is a letter code (handled in 1.2.2). pandas does not recognise `\N` as missing, so the data *looks* complete. This is why sentinels must be replaced before anything else.

#### 1.2.2 Discover the placeholder tokens in every table

1.2.1 found `\N` in one table. To avoid assuming it is the only placeholder, we scan **all 14 tables**. Any short (≤ 2 characters) non-numeric value is collected, together with how many different columns it appears in.

**Decision rule:** a placeholder is a token that means "nothing" wherever it appears. It therefore shows up in many unrelated columns, or it is empty. A real category code (such as `R` in `positionText`) only lives in one kind of column.

In [25]:
token_rows = []
for name in TABLE_NAMES:
    for col in raw[name].columns:
        values = raw[name][col]
        is_number = pd.to_numeric(values, errors='coerce').notna()
        short_text = values[(~is_number) & (values.str.len() <= 2)]
        for token, count in short_text.value_counts().items():
            token_rows.append({'token': repr(token), 'table': name, 'column': col, 'count': count})

tokens = pd.DataFrame(token_rows)
token_summary = tokens.groupby('token').agg(total_count=('count', 'sum'),
                                            n_columns=('column', 'count'),
                                            example_column=('column', 'first'))
token_summary = token_summary.sort_values('n_columns', ascending=False)
token_summary

Error: No connection selected.

In [26]:
plt.figure(figsize=(9, 4))
plt.bar(token_summary.index, token_summary['n_columns'], color='grey')
plt.title('Short non-numeric tokens: number of different columns each appears in')
plt.xlabel('Token')
plt.ylabel('Columns')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_token_discovery.png')
plt.show()

Error: No connection selected.

**Decision, from the table:**
* **`'\N'`** appears in **30 different columns** across many tables (IDs, times, dates, codes), far more than any other token (≤ 3 columns). It is the dataset's missing-value marker.
* **`''`** (empty string) appears in **2 columns**. An empty cell carries no value by definition, so it is also treated as missing.
* All other tokens are real values that belong to one kind of column:
  * letter codes in the `positionText` columns (`R`, `F`, `W`, `N`, `D`, `E`) and in `constructor_results.status` (`D`), which are explained by the `status` table;
  * `UK` in `circuits.country`;
  * short first names (`Jo`, `Al`, `Ed`) in `drivers.forename`;
  * short team names and references (`RE`, `rb`, `re`) in `constructors`.

  These are **kept**.

Next, we count the two placeholders per column.

In [27]:
sentinel_rows = []
for name in TABLE_NAMES:
    table = raw[name]
    for col in table.columns:
        n_backslash_n = (table[col] == '\\N').sum()
        n_empty = (table[col] == '').sum()
        if n_backslash_n > 0 or n_empty > 0:
            sentinel_rows.append({'table': name, 'column': col,
                                  r"'\N' count": n_backslash_n, "'' count": n_empty,
                                  '% of rows': round(100 * (n_backslash_n + n_empty) / len(table), 1)})

sentinels = pd.DataFrame(sentinel_rows)
sentinels

Error: No connection selected.

In [28]:
plot_df = sentinels.copy()
plot_df['label'] = plot_df['table'] + '.' + plot_df['column']
plot_df = plot_df.sort_values('% of rows')

plt.figure(figsize=(9, 7))
plt.barh(plot_df['label'], plot_df['% of rows'], color='grey')
plt.title('Raw data: % of rows holding a missing-value placeholder')
plt.xlabel('% of rows')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_sentinels.png')
plt.show()

Error: No connection selected.

**Observations**

* `\N` appears in 6 tables. Whether it means "value does not exist" or "value was lost" is checked in 1.5.2 with a missingness-by-season analysis.
* `qualifying.q2` and `qualifying.q3` also contain **empty strings**. Replacing only `\N` would leave these behind, so both tokens must be treated as missing.
* The `races` session columns (`fp1_date` … `sprint_time`) are > 90 % missing because the dataset only records practice-session schedules for recent seasons.

#### 1.2.3 Podium base rate over time

The target is `positionOrder <= 3`. Its share of rows (the **base rate**) tells us how imbalanced the classification problem is. That matters for choosing metrics later; Lab 2 showed accuracy can mislead when classes are unbalanced.

In [29]:
# Small typed view of results joined with the season year (no cleaning yet)
res_year = raw['results'].merge(raw['races'][['raceId', 'year']], on='raceId')
res_year['year'] = res_year['year'].astype(int)
res_year['grid'] = res_year['grid'].astype(int)
res_year['positionOrder'] = res_year['positionOrder'].astype(int)
res_year['podium'] = (res_year['positionOrder'] <= 3).astype(int)

rows_per_season = res_year.groupby('year')['podium'].count()
podium_rate_raw = res_year.groupby('year')['podium'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(rows_per_season.index, rows_per_season.values, color='grey')
axes[0].set_title('Raw results: rows per season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Result rows')

axes[1].plot(podium_rate_raw.index, podium_rate_raw.values, color='crimson', linewidth=2)
axes[1].set_title('Raw results: podium rate per season')
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Podium rate (%)')

plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_base_rate_raw.png')
plt.show()

print(f"Overall podium rate (raw): {res_year['podium'].mean():.3f}")

Error: No connection selected.

**Observation:** only about **1 in 8** rows is a podium, so the positive class is the minority. The rate dips to ≈ 8 % around 1989, exactly when the number of rows per season peaks. More entries cannot create fewer podiums, so section 1.2.4 investigates what those extra rows are.

#### 1.2.4 What does `grid = 0` mean?

`grid` is the starting position. A grid of `0` is not a real position, so we use the `status` table to see what these rows are.

In [30]:
res_status = res_year.merge(raw['status'], on='statusId')
grid_zero = res_status[res_status['grid'] == 0].copy()
print('Rows with grid = 0:', len(grid_zero))
grid_zero['status'].value_counts().head(10)

Error: No connection selected.

In [31]:
# Keep the most common statuses, group the rest as "Other"
top_status = grid_zero['status'].value_counts().head(6).index
grid_zero['decade'] = (grid_zero['year'] // 10) * 10
grid_zero['status_group'] = grid_zero['status']
grid_zero.loc[~grid_zero['status'].isin(top_status), 'status_group'] = 'Other'

pivot = pd.pivot_table(grid_zero, index='decade', columns='status_group',
                       values='raceId', aggfunc='count', fill_value=0)

pivot.plot(kind='bar', stacked=True, figsize=(11, 4.5), colormap='tab10')
plt.title('Raw results: rows with grid = 0, by decade and status')
plt.xlabel('Decade')
plt.ylabel('Rows')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_grid_zero.png')
plt.show()

Error: No connection selected.

To investigate the late-1980s result-row peak, we compare all result rows per race with rows carrying a positive stored grid position. A positive grid is a useful descriptive proxy here, but it does not by itself prove an actual race start. The later participation audit tests this distinction directly.

In [32]:
per_race = res_year.groupby(['year', 'raceId']).agg(
    entrants=('driverId', 'count'),
    positive_grid=('grid', lambda g: (g > 0).sum()))
per_season = per_race.groupby('year').mean()

plt.figure(figsize=(11, 4))
plt.plot(per_season.index, per_season['entrants'], color='grey', linewidth=2,
         label='result rows per race')
plt.plot(per_season.index, per_season['positive_grid'], color='steelblue', linewidth=2,
         label='positive-grid rows per race')
plt.title('Average result rows vs. positive-grid rows per race')
plt.xlabel('Season')
plt.ylabel('Drivers per race')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_entrants_vs_starters.png')
plt.show()

peak = per_season['entrants'].idxmax()
print(f"Peak season {peak}: {per_season.loc[peak, 'entrants']:.1f} result rows vs "
      f"{per_season.loc[peak, 'positive_grid']:.1f} positive-grid rows per race")

Error: No connection selected.

`grid = 0` combines qualifying eliminations, withdrawals, and some drivers who raced. A positive grid can also remain on a row for a driver who did not start. The chart above measures a recorded grid code, not verified participation. Section 1.4.4 uses completed laps and other recorded evidence to classify starts and leaves unresolved cases visible.

#### 1.2.5 Relationship between recorded grid and podium

This descriptive check uses result rows with a positive recorded grid position from 1 through 20. It does not use grid as independent proof of a start.

In [33]:
positive_grid_rows = res_year[(res_year['grid'] > 0) & (res_year['grid'] <= 20)]
podium_by_grid = positive_grid_rows.groupby('grid')['podium'].mean() * 100

plt.figure(figsize=(9, 4))
plt.bar(podium_by_grid.index, podium_by_grid.values, color='steelblue')
plt.title('Raw results: podium rate by recorded grid position')
plt.xlabel('Grid position')
plt.ylabel('Podium rate (%)')
plt.xticks(range(1, 21))
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_podium_by_grid.png')
plt.show()
podium_by_grid.head(10).round(1)

Error: No connection selected.

**Observation:** podium rate falls steeply and non-linearly with grid position, from ≈ 64 % on pole to under 10 % from P8. Grid will be a key feature; the curved shape suggests tree models or a transformed grid feature may fit it better than a single linear term.

#### 1.2.6 Qualifying coverage by season

The `qualifying` table is the only source of qualifying position and lap times. We measure, per season, the share of result rows that have a matching qualifying row.

In [34]:
quali_keys = raw['qualifying']['raceId'] + '_' + raw['qualifying']['driverId']
res_year['has_quali'] = (res_year['raceId'] + '_' + res_year['driverId']).isin(quali_keys)

quali_coverage = res_year.groupby('year')['has_quali'].mean() * 100

# Thresholds computed from the data, not read off the plot:
# first season with any qualifying rows, and first season from which EVERY later season is >= 95 %
# (95 % rather than 100 % allows for the odd driver per season with no qualifying row;
#  the lowest season from 2003 on is printed so the margin is visible)
first_year = quali_coverage[quali_coverage > 0].index.min()
full_from = min(y for y in quali_coverage.index if (quali_coverage.loc[y:] >= 95).all())
print('First season with qualifying rows   :', first_year)
print('Coverage >= 95 % in every season from:', full_from)
print(f'Lowest coverage from that season on : {quali_coverage.loc[full_from:].min():.1f} %')

plt.figure(figsize=(11, 3.8))
plt.plot(quali_coverage.index, quali_coverage.values, color='steelblue', linewidth=2)
plt.axvline(first_year, color='black', linestyle='--', linewidth=1)
plt.axvline(full_from, color='black', linestyle='--', linewidth=1)
plt.text(first_year + 0.5, 55, 'first qualifying rows', fontsize=9)
plt.text(full_from + 0.5, 30, 'full coverage from here', fontsize=9)
plt.title('Share of result rows that have a qualifying row, per season')
plt.xlabel('Season')
plt.ylabel('Coverage (%)')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_2_quali_coverage.png')
plt.show()

quali_coverage.loc[1993:2004].round(1).to_frame('coverage %').T

Error: No connection selected.

**Observation:** qualifying coverage varies sharply by era. Qualifying remains useful for Q1's required distinction between qualifying position and starting grid. Following confirmed TA guidance, it is omitted from the prediction inputs, so sparse historical qualifying coverage does not exclude pre-2003 race entries.


### 1.3 Key audit

#### 1.3.1 Primary keys

A **primary key** identifies each row and must be **unique** and **never missing**. Most tables have an ID column. `lap_times` and `pit_stops` have none, so their key is a combination of columns.

In [35]:
primary_keys = {
    'circuits': ['circuitId'], 'constructors': ['constructorId'], 'drivers': ['driverId'],
    'races': ['raceId'], 'seasons': ['year'], 'status': ['statusId'],
    'results': ['resultId'], 'qualifying': ['qualifyId'], 'sprint_results': ['resultId'],
    'driver_standings': ['driverStandingsId'], 'constructor_standings': ['constructorStandingsId'],
    'constructor_results': ['constructorResultsId'],
    'lap_times': ['raceId', 'driverId', 'lap'],     # composite key
    'pit_stops': ['raceId', 'driverId', 'stop'],    # composite key
}

pk_rows = []
for name, key in primary_keys.items():
    table = raw[name]
    missing = table[key].isin(['\\N', '']).any(axis=1).sum()   # sentinel counts as missing
    duplicated = table.duplicated(subset=key).sum()
    pk_rows.append({'table': name, 'primary key': ' + '.join(key),
                    'missing keys': missing, 'duplicate keys': duplicated,
                    'valid': missing == 0 and duplicated == 0})

pk_table = pd.DataFrame(pk_rows)
display(pk_table)
assert pk_table['valid'].all(), 'Primary-key audit failed; inspect pk_table before continuing.'


Error: No connection selected.

**Result:** all 14 primary keys are unique and complete.

#### 1.3.2 Foreign keys

A **foreign key** points to a primary key in another table, e.g. `results.driverId → drivers.driverId`. Every value must exist in the table it points to; otherwise joins would silently lose rows or create empty values.

In [36]:
# (child table, child column, parent table, parent column)
foreign_keys = [
    ('races', 'circuitId', 'circuits', 'circuitId'),
    ('races', 'year', 'seasons', 'year'),
    ('results', 'raceId', 'races', 'raceId'),
    ('results', 'driverId', 'drivers', 'driverId'),
    ('results', 'constructorId', 'constructors', 'constructorId'),
    ('results', 'statusId', 'status', 'statusId'),
    ('qualifying', 'raceId', 'races', 'raceId'),
    ('qualifying', 'driverId', 'drivers', 'driverId'),
    ('qualifying', 'constructorId', 'constructors', 'constructorId'),
    ('sprint_results', 'raceId', 'races', 'raceId'),
    ('sprint_results', 'driverId', 'drivers', 'driverId'),
    ('sprint_results', 'constructorId', 'constructors', 'constructorId'),
    ('sprint_results', 'statusId', 'status', 'statusId'),
    ('lap_times', 'raceId', 'races', 'raceId'),
    ('lap_times', 'driverId', 'drivers', 'driverId'),
    ('pit_stops', 'raceId', 'races', 'raceId'),
    ('pit_stops', 'driverId', 'drivers', 'driverId'),
    ('driver_standings', 'raceId', 'races', 'raceId'),
    ('driver_standings', 'driverId', 'drivers', 'driverId'),
    ('constructor_standings', 'raceId', 'races', 'raceId'),
    ('constructor_standings', 'constructorId', 'constructors', 'constructorId'),
    ('constructor_results', 'raceId', 'races', 'raceId'),
    ('constructor_results', 'constructorId', 'constructors', 'constructorId'),
]

fk_rows = []
for child, child_col, parent, parent_col in foreign_keys:
    orphans = (~raw[child][child_col].isin(raw[parent][parent_col])).sum()
    fk_rows.append({'foreign key': f'{child}.{child_col}', 'references': f'{parent}.{parent_col}',
                    'orphan rows': orphans})

fk_table = pd.DataFrame(fk_rows)
print('Total orphan rows:', fk_table['orphan rows'].sum())
assert fk_table['orphan rows'].eq(0).all(), 'Foreign-key audit failed; inspect fk_table before continuing.'
fk_table


Error: No connection selected.

Race-level tables such as lap times, pit stops and qualifying should also only describe drivers who actually entered that race, i.e. `(raceId, driverId)` pairs that exist in `results`.

In [37]:
result_pairs = raw['results']['raceId'] + '_' + raw['results']['driverId']

for name in ['qualifying', 'lap_times', 'pit_stops', 'sprint_results', 'driver_standings']:
    pairs = (raw[name]['raceId'] + '_' + raw[name]['driverId']).drop_duplicates()
    print(f'{name:<18} pairs not found in results: {(~pairs.isin(result_pairs)).sum()}')

Error: No connection selected.

**Result:** all foreign keys reconcile, with zero orphans.

`driver_standings` has ≈ 8.7 k pairs not in `results`.

**Hypothesis:** the championship table after each round still lists a driver who missed that race, carrying over their earlier points.

**Test:** if that is true, every such driver must have raced **earlier in the same season**.

In [38]:
races_small = raw['races'][['raceId', 'year', 'round']].astype(int)
standings = raw['driver_standings'][['raceId', 'driverId']].astype(int).merge(races_small, on='raceId')
entries = raw['results'][['raceId', 'driverId']].astype(int).merge(races_small, on='raceId')

# Standings rows whose (race, driver) pair is NOT in results
entry_keys = entries['raceId'].astype(str) + '_' + entries['driverId'].astype(str)
standing_keys = standings['raceId'].astype(str) + '_' + standings['driverId'].astype(str)
not_entered = standings[~standing_keys.isin(entry_keys)]

# First round each driver raced in each season
first_round = entries.groupby(['year', 'driverId'])['round'].min().rename('first_round').reset_index()
check = not_entered.merge(first_round, on=['year', 'driverId'], how='left')
raced_earlier = (check['round'] > check['first_round']).mean() * 100

print('Standings rows for drivers not in that race:', len(check))
print(f'... of which the driver raced earlier that season: {raced_earlier:.1f} %')

Error: No connection selected.

**Result:** every standings row without a corresponding current-race result belongs to a driver with an earlier result in that season. This is consistent with championship standings carrying forward, so a standings row alone cannot establish current-race participation.

#### 1.3.3 Is `(raceId, driverId)` unique in `results`?

`resultId` is unique, but the later analyses need one row per driver per race. The next check tests that grain across the full results table.

In [39]:
dup_mask = raw['results'].duplicated(subset=['raceId', 'driverId'], keep=False)
duplicates = raw['results'][dup_mask].merge(raw['races'][['raceId', 'year']], on='raceId') \
                                     .merge(raw['status'], on='statusId')

print('Duplicate rows:', len(duplicates))
print('Affected (raceId, driverId) pairs:', duplicates.groupby(['raceId', 'driverId']).ngroups)
print('Latest season with a duplicate:', duplicates['year'].astype(int).max())

Error: No connection selected.

To characterize duplicate `(raceId, driverId)` pairs, the next cell counts how many rows in each pair have positive completed laps. The labels describe recorded values only; they do not infer the historical cause of a duplicate.

In [40]:
duplicates['laps'] = duplicates['laps'].astype(int)
positive_lap_rows = duplicates.groupby(['raceId', 'driverId'])['laps'].apply(lambda laps: int(laps.gt(0).sum()))
pair_type = positive_lap_rows.map(
    lambda n: 'multiple_positive_lap_rows' if n >= 2 else 'at_most_one_positive_lap_row'
).rename('dup_type')
duplicates = duplicates.merge(pair_type.reset_index(), on=['raceId', 'driverId'])

pair_table = duplicates.drop_duplicates(['raceId', 'driverId'])
print('Duplicate pairs by recorded lap evidence:')
print(pair_table['dup_type'].value_counts().to_string())

pivot = pd.pivot_table(pair_table, index='year', columns='dup_type', values='driverId',
                       aggfunc='count', fill_value=0)
pivot.plot(kind='bar', stacked=True, figsize=(10, 3.5), color=['grey', 'crimson'])
plt.title('Duplicate driver-race pairs by year and lap evidence')
plt.xlabel('Season')
plt.ylabel('Pairs')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_3_duplicates.png')
plt.show()

Error: No connection selected.

In [41]:
pair_summary = pair_table.groupby('dup_type').agg(
    affected_pairs=('driverId', 'size'),
    earliest_year=('year', 'min'),
    latest_year=('year', 'max'),
)
print(pair_summary.to_string())

Error: No connection selected.

The full-table check finds 85 duplicate driver-race pairs (176 result rows), all dated 1950–1978. The aggregate table separates pairs with multiple positive-lap rows from pairs with at most one. It does not establish why each duplicate was recorded. Section 1.4.5 evaluates a deterministic one-row-per-driver-race rule.

### 1.4 Cleaning

#### 1.4.1 Replace sentinels with real missing values

Both `\N` and `''` are replaced with `NaN`, the value pandas understands as missing.

In [ ]:
for name in TABLE_NAMES:
    data[name] = data[name].replace(['\\N', ''], np.nan)

# Check: no sentinel token may remain anywhere
remaining = sum(data[name].isin(['\\N', '']).sum().sum() for name in TABLE_NAMES)
print('Sentinel tokens before:', sentinels[[r"'\N' count", "'' count"]].sum().sum())
print('Sentinel tokens after :', remaining)

#### 1.4.2 Normalize categorical labels

Lab 1 calls for programmatic checks of categorical inconsistencies. The next cells test **every distinct value** for surrounding whitespace, similar stems, and geographic overlap between country-label groups. These checks flag candidate aliases; geographic overlap alone does not prove equivalence. The declared category mapping is applied consistently to all rows and its coverage is checked afterward. No individual race or driver record is used to select a correction.

In [ ]:
for table, col in [('circuits', 'country'), ('drivers', 'nationality')]:
    values = pd.Series(data[table][col].unique())

    # Check 1: hidden leading/trailing spaces
    print(f'{table}.{col} - values with hidden spaces:', values[values != values.str.strip()].tolist())

    # Check 2: same first 6 letters (single-word values only)
    single = values.str.strip()
    single = single[~single.str.contains('-')]
    groups = single.groupby(single.str[:6].str.lower()).apply(list)
    print(f'{table}.{col} - values sharing their first 6 letters:', groups[groups.apply(len) > 1].tolist())

In [ ]:
# Check 3: is every circuit of label A inside the lat/lng box of label B?
circ = data['circuits'].copy()
circ['lat'] = circ['lat'].astype(float)
circ['lng'] = circ['lng'].astype(float)
box = circ.groupby('country').agg(lat_min=('lat', 'min'), lat_max=('lat', 'max'),
                                  lng_min=('lng', 'min'), lng_max=('lng', 'max'), n=('lat', 'count'))

flagged = []
for a in box.index:
    for b in box.index:
        if a == b or box.loc[b, 'n'] < 2:        # a box needs at least 2 circuits
            continue
        a_circuits = circ[circ['country'] == a]
        inside = (a_circuits['lat'].between(box.loc[b, 'lat_min'], box.loc[b, 'lat_max']) &
                  a_circuits['lng'].between(box.loc[b, 'lng_min'], box.loc[b, 'lng_max']))
        if inside.all():
            flagged.append((a, b))
print('Country labels fully inside another label\'s area:', flagged)

In [ ]:
# Visualize every country-label pair flagged by the table-wide geographic test.
if flagged:
    fig, axes = plt.subplots(len(flagged), 1, figsize=(8, 3.8 * len(flagged)), squeeze=False)
    for ax, (a, b) in zip(axes[:, 0], flagged):
        for label, color in [(b, 'steelblue'), (a, 'crimson')]:
            pts = circ[circ['country'] == label]
            ax.scatter(pts['lng'], pts['lat'], color=color, s=60,
                       label=f"'{label}' ({len(pts)} circuits)")
        ax.set_title(f"Geographic overlap candidate: '{a}' and '{b}'")
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.legend()
    fig.tight_layout()
    fig.savefig(f'{FIG_PATH}/1_4_country_variants.png')
    plt.show()
else:
    print('No geographic-overlap candidates found.')

**Category checks:** the code reports whitespace variants, same-stem nationality labels, and geographically overlapping country labels across the full category sets. In this snapshot it flags `Argentinian `, the `Argentine`/`Argentinian` pair, and `United States`/`USA`. The next cell declares those label aliases explicitly and applies them to all records. Coordinates provide a candidate check, not proof that two labels mean the same country; the normalization decision uses the category names, as in Lab 1. The code reports the before/after number of categories.

In [ ]:
data['circuits']['country'] = data['circuits']['country'].str.strip().replace({'United States': 'USA'})
data['drivers']['nationality'] = data['drivers']['nationality'].str.strip().replace({'Argentinian': 'Argentine'})

print('Circuit countries   :', raw['circuits']['country'].nunique(), '->', data['circuits']['country'].nunique())
print('Driver nationalities:', raw['drivers']['nationality'].nunique(), '->', data['drivers']['nationality'].nunique())

#### 1.4.3 Convert columns to their correct types

The code converts ID and count fields to numbers, calendar fields to dates, and consistently formatted duration fields to seconds. Each choice applies to the full column and is followed by coverage or error checks. The mixed-format `results.time` text is left unchanged; it is a current-race outcome field and is not a planned pre-race predictor.

In [ ]:
# Columns that should be numbers (pd.to_numeric turns '7' into 7 and keeps NaN as NaN)
numeric_columns = {
    'circuits': ['circuitId', 'lat', 'lng', 'alt'],
    'constructors': ['constructorId'],
    'drivers': ['driverId', 'number'],
    'races': ['raceId', 'year', 'round', 'circuitId'],
    'seasons': ['year'],
    'status': ['statusId'],
    'results': ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position',
                'positionOrder', 'points', 'laps', 'milliseconds', 'fastestLap', 'rank',
                'fastestLapSpeed', 'statusId'],
    'sprint_results': ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
                       'position', 'positionOrder', 'points', 'laps', 'milliseconds',
                       'fastestLap', 'statusId'],
    'qualifying': ['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position'],
    'lap_times': ['raceId', 'driverId', 'lap', 'position', 'milliseconds'],
    'pit_stops': ['raceId', 'driverId', 'stop', 'lap', 'milliseconds'],
    'driver_standings': ['driverStandingsId', 'raceId', 'driverId', 'points', 'position', 'wins'],
    'constructor_standings': ['constructorStandingsId', 'raceId', 'constructorId', 'points',
                              'position', 'wins'],
    'constructor_results': ['constructorResultsId', 'raceId', 'constructorId', 'points'],
}

for name, columns in numeric_columns.items():
    for col in columns:
        data[name][col] = pd.to_numeric(data[name][col])

In [ ]:
# Parse all race dates and birth dates, then report table-wide validity.
data['races']['date'] = pd.to_datetime(data['races']['date'], errors='coerce')
data['drivers']['dob'] = pd.to_datetime(data['drivers']['dob'], errors='coerce')
print('Race dates missing after parsing:', data['races']['date'].isna().sum())
print('Driver birth dates missing after parsing:', data['drivers']['dob'].isna().sum())
print('Race-date range:', data['races']['date'].min(), 'to', data['races']['date'].max())

**Durations.** Before writing a converter we need to know which text formats actually occur. Each value's *shape* is computed by replacing every digit with `9`, so `1:26.572` becomes `9:99.999`.

In [ ]:
duration_columns = [('qualifying', 'q1'), ('qualifying', 'q2'), ('qualifying', 'q3'),
                    ('lap_times', 'time'), ('pit_stops', 'duration'), ('results', 'fastestLapTime')]

shape_rows = []
for table, col in duration_columns:
    shapes = data[table][col].dropna().str.replace(r'[0-9]', '9', regex=True).value_counts()
    for shape, count in shapes.items():
        shape_rows.append({'column': f'{table}.{col}', 'shape': shape, 'count': count})

pd.DataFrame(shape_rows)

**Observation:** the table shows three formats: `ss.sss` (pit stops), `m:ss.sss` (most laps) and `h:mm:ss.sss` (43 very long laps). The minutes field has one or two digits. The function below handles all of them: it splits on `:` and reads the pieces from the right, as seconds, then minutes × 60, then hours × 3600.

In [ ]:
def time_to_seconds(value):
    # Missing stays missing
    if pd.isna(value):
        return np.nan
    parts = str(value).split(':')          # '1:26.572' -> ['1', '26.572']
    seconds = 0.0
    for power, part in enumerate(reversed(parts)):
        seconds += float(part) * (60 ** power)   # seconds*1 + minutes*60 + hours*3600
    return seconds

print(time_to_seconds('23.456'), time_to_seconds('1:26.572'), time_to_seconds('1:02:03.500'))

In [ ]:
data['qualifying']['q1_s'] = data['qualifying']['q1'].apply(time_to_seconds)
data['qualifying']['q2_s'] = data['qualifying']['q2'].apply(time_to_seconds)
data['qualifying']['q3_s'] = data['qualifying']['q3'].apply(time_to_seconds)
data['lap_times']['time_s'] = data['lap_times']['time'].apply(time_to_seconds)
data['pit_stops']['duration_s'] = data['pit_stops']['duration'].apply(time_to_seconds)
data['results']['fastestLapTime_s'] = data['results']['fastestLapTime'].apply(time_to_seconds)

print('Converted nonmissing durations:')
print(pd.Series({
    'qualifying.q1_s': data['qualifying']['q1_s'].notna().sum(),
    'qualifying.q2_s': data['qualifying']['q2_s'].notna().sum(),
    'qualifying.q3_s': data['qualifying']['q3_s'].notna().sum(),
    'lap_times.time_s': data['lap_times']['time_s'].notna().sum(),
    'pit_stops.duration_s': data['pit_stops']['duration_s'].notna().sum(),
    'results.fastestLapTime_s': data['results']['fastestLapTime_s'].notna().sum(),
}).to_string())

**Checking duration conversion:** `lap_times` and `pit_stops` also store each duration in milliseconds. The next cell compares converted seconds with that independent numeric field for every row and reports the largest error and the count outside tolerance.

In [ ]:
lap_error = (data['lap_times']['time_s'] - data['lap_times']['milliseconds'] / 1000).abs()
pit_error = (data['pit_stops']['duration_s'] - data['pit_stops']['milliseconds'] / 1000).abs()

print(f'lap_times : max error = {lap_error.max():.6f} s, rows wrong by > 1 ms = {(lap_error > 0.001).sum()}')
print(f'pit_stops : max error = {pit_error.max():.6f} s, rows wrong by > 1 ms = {(pit_error > 0.001).sum()}')

**Result:** the duration conversions match the stored millisecond values within the reported tolerance.

#### 1.4.4 Classify participation from recorded evidence

The next table summarizes completed-lap evidence by every status label. A positive completed-lap count supports a start. Zero completed laps alone cannot distinguish a very early retirement from a non-start, so that distinction remains unresolved unless another recorded field provides evidence.

In [ ]:
res_status_clean = data['results'].merge(data['status'], on='statusId')
zero_laps = res_status_clean.groupby('status').agg(rows=('laps', 'count'),
                                                   pct_zero_laps=('laps', lambda l: 100 * (l == 0).mean()))
zero_laps = zero_laps[zero_laps['pct_zero_laps'] >= 90].sort_values('rows', ascending=False)
zero_laps.round(1)

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(zero_laps.index, zero_laps['rows'], color='grey')
plt.title('Statuses where >= 90 % of rows completed zero laps')
plt.xlabel('Rows')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_4_zero_lap_statuses.png')
plt.show()

big = zero_laps[zero_laps['rows'] >= 100]
print('Rows covered by the statuses with >= 100 rows:', big['rows'].sum())
print('Rows in all other zero-lap statuses:', zero_laps['rows'].sum() - big['rows'].sum())

**Participation rule used below:** a positive number of completed race laps confirms a start for this conservative analysis. **Every zero-lap row remains unresolved**, regardless of its status label or other table entries. A zero-lap record could be a very early retirement or a non-start; this notebook does not decide which. The rule is applied to every row, with a full status-by-evidence table and plot below. `grid_clean` treats zero as unavailable numeric position. `grid_zero_with_positive_laps` identifies grid-zero rows with positive completed laps, without assigning a pre-race reason to that code.

In [ ]:
# Positive completed laps confirm a start for this conservative analysis.
# Every zero-lap row remains unresolved, regardless of its status label.

for name in ['results', 'sprint_results']:
    table = data[name]
    observed_start = table['laps'].gt(0)

    table['entry_type'] = 'unresolved_start'
    table.loc[observed_start, 'entry_type'] = 'confirmed_start'
    table['grid_clean'] = table['grid'].replace(0, np.nan)
    table['grid_zero_with_positive_laps'] = table['grid'].eq(0) & table['entry_type'].eq('confirmed_start')

data['results']['grid_is_zero'] = data['results']['grid'].eq(0)
start_audit = data['results'].merge(data['status'], on='statusId')
start_audit['completed_laps'] = start_audit['laps'].gt(0)
print('Race participation evidence:')
print(pd.crosstab(start_audit['entry_type'], start_audit['grid_is_zero'], margins=True))
status_evidence = start_audit.groupby('status').agg(
    result_rows=('resultId', 'size'),
    positive_lap_rows=('completed_laps', 'sum'),
    confirmed_starts=('entry_type', lambda x: x.eq('confirmed_start').sum()),
    unresolved_starts=('entry_type', lambda x: x.eq('unresolved_start').sum()),
).sort_values('result_rows', ascending=False)
print('All status labels by recorded participation evidence:')
print(status_evidence.to_string())

start_plot = pd.crosstab(start_audit['entry_type'], start_audit['grid_is_zero'])
start_plot.plot(kind='bar', stacked=True, color=['steelblue', 'grey'], figsize=(8, 4))
plt.title('Participation evidence by stored grid code')
plt.xlabel('Participation evidence')
plt.ylabel('Result rows')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_4_participation_evidence.png')
plt.show()

The table above reports **every status label** against completed-lap evidence. No status name is used to classify a zero-lap record as a definite start or non-start. All zero-lap records remain unresolved, and starter-based answers below use only positive-lap starts for their point estimates.

#### 1.4.5 One row per driver-race: separate the label from pre-race inputs

Compare label preservation under first-row, most-laps and best-finish rules. The
selected post-race descriptive view keeps the best finish (laps break ties), but
**no prediction input is selected using that rule**. The driver-race target is the
maximum podium indicator over all source records.

Independently resolve starting inputs using only `(raceId, driverId, grid,
constructorId)`. A single recorded positive grid is retained; zero-only records
retain code zero; conflicting positive grids become missing. Constructor identity
is retained only when all nonmissing records agree. Ambiguity is preserved rather
than choosing the car that happened to finish best. All driver-race entries and
earlier years remain. The descriptive view uses this same resolved driver grid.


In [ ]:
dup_rows = data['results'][data['results'].duplicated(subset=['raceId', 'driverId'], keep=False)]
pairs_with_podium = dup_rows[dup_rows['positionOrder'] <= 3][['raceId', 'driverId']].drop_duplicates()

rules = {
    'first row':   dup_rows.sort_values('resultId'),
    'most laps':   dup_rows.sort_values('laps', ascending=False),
    'best result': dup_rows.sort_values(['positionOrder', 'laps'], ascending=[True, False]),
}

# Pairs where both rows have the same lap count: 'most laps' cannot decide there and falls back to file order
laps_tied = dup_rows.groupby(['raceId', 'driverId'])['laps'].nunique().eq(1).sum()

rule_rows = []
for rule_name, ordered in rules.items():
    kept = ordered.drop_duplicates(subset=['raceId', 'driverId'], keep='first')
    podiums_kept = (kept['positionOrder'] <= 3).sum()
    rule_rows.append({'rule': rule_name, 'real podiums in duplicated pairs': len(pairs_with_podium),
                      'podiums kept': podiums_kept, 'podiums lost': len(pairs_with_podium) - podiums_kept,
                      'pairs the rule cannot decide': laps_tied if rule_name == 'most laps' else 0})

rule_compare = pd.DataFrame(rule_rows)
rule_compare

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.bar(rule_compare['rule'], rule_compare['podiums lost'], color=['grey', 'grey', 'steelblue'])
plt.title('Real podiums lost by each de-duplication rule')
plt.ylabel('Podiums lost')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_4_dedup_rules.png')
plt.show()

The best-finish rule preserves the driver-race label, not a reliable pre-race car
assignment. The next cell creates two separate objects: `results_dedup`, a
post-race descriptive representative, and `pre_race_entries`, an input-only
resolution. Modeling uses the second object plus separately aggregated labels.
Known historical constructor outcomes are aggregated from all original records
only for **later** constructor-form features.


In [ ]:
results = data['results']
rows_before = len(results)
results_all_records = results.copy(deep=True)

def build_pre_race_entries(records):
    allowed = records[['raceId', 'driverId', 'grid', 'constructorId']].copy()
    keys = ['raceId', 'driverId']
    def resolved_grid(values):
        positive = pd.unique(values.dropna()[values.dropna().gt(0)])
        if len(positive) == 1:
            return float(positive[0])
        if len(positive) > 1:
            return np.nan
        return 0.0 if values.eq(0).any() else np.nan
    def agreed_constructor(values):
        unique = pd.unique(values.dropna())
        return float(unique[0]) if len(unique) == 1 else np.nan
    grouped = allowed.groupby(keys, sort=True)
    resolved = grouped.agg(grid=('grid', resolved_grid),
        constructorId=('constructorId', agreed_constructor),
        source_records=('grid', 'size')).reset_index()
    resolved['grid_conflict'] = grouped['grid'].apply(
        lambda v: v[v.gt(0)].nunique() > 1).to_numpy()
    resolved['constructor_conflict'] = grouped['constructorId'].nunique().gt(1).to_numpy()
    assert not resolved.duplicated(keys).any()
    return resolved

pre_race_entries = build_pre_race_entries(results_all_records)
driver_race_labels = (results_all_records.assign(podium=results_all_records['positionOrder'].le(3).astype(int))
    .groupby(['raceId', 'driverId'], as_index=False)['podium'].max())
duplicate_input_audit = pre_race_entries[pre_race_entries['source_records'].gt(1)].copy()
results['duplicate_result_pair'] = results.duplicated(['raceId', 'driverId'], keep=False).astype(int)
results_dedup = (results.sort_values(['raceId', 'driverId', 'positionOrder', 'laps', 'resultId'],
                                    ascending=[True, True, True, False, True])
    .drop_duplicates(['raceId', 'driverId']).sort_values('resultId').reset_index(drop=True))
results_dedup['representative_result_grid'] = results_dedup['grid']
grid_lookup = pre_race_entries.set_index(['raceId', 'driverId'])['grid']
keys = pd.MultiIndex.from_frame(results_dedup[['raceId', 'driverId']])
results_dedup['grid'] = grid_lookup.reindex(keys).to_numpy()
results_dedup['grid_clean'] = results_dedup['grid'].where(results_dedup['grid'].gt(0))
results_dedup['grid_is_zero'] = results_dedup['grid'].eq(0)
results_dedup['grid_zero_with_positive_laps'] = results_dedup['grid_is_zero'] & results_dedup['entry_type'].eq('confirmed_start')
print('Rows before / after:', rows_before, '/', len(results_dedup))
print('Extra records removed:', rows_before - len(results_dedup))
print('Ambiguous positive grids:', int(pre_race_entries['grid_conflict'].sum()),
      '| ambiguous constructors:', int(pre_race_entries['constructor_conflict'].sum()))
display(duplicate_input_audit)


In [ ]:
assert not results_dedup.duplicated(['raceId', 'driverId']).any()
podium_before = results_all_records.loc[results_all_records['positionOrder'].le(3), ['raceId', 'driverId']].drop_duplicates()
podium_after = results_dedup.loc[results_dedup['positionOrder'].le(3), ['raceId', 'driverId']]
assert set(map(tuple, podium_before.to_numpy())) == set(map(tuple, podium_after.to_numpy()))
label_check = results_dedup[['raceId', 'driverId', 'positionOrder']].merge(
    driver_race_labels, on=['raceId', 'driverId'], validate='one_to_one')
assert label_check['positionOrder'].le(3).astype(int).equals(label_check['podium'])
# Change every forbidden outcome field; input construction must be invariant.
tampered = results_all_records.copy(deep=True)
for column in ['positionOrder', 'laps', 'points', 'statusId']:
    tampered[column] = np.arange(len(tampered), dtype=float)[::-1] + 999
pd.testing.assert_frame_equal(pre_race_entries, build_pre_race_entries(tampered))
# Swapping finishing ranks within duplicate pairs preserves aggregate targets too.
tampered['positionOrder'] = results_all_records['positionOrder'].copy()
for _, group in tampered[tampered.duplicated(['raceId', 'driverId'], keep=False)].groupby(['raceId', 'driverId']):
    idx = group.index
    tampered.loc[idx, 'positionOrder'] = results_all_records.loc[idx, 'positionOrder'].to_numpy()[::-1]
swapped_labels = (tampered.assign(podium=tampered['positionOrder'].le(3).astype(int))
    .groupby(['raceId', 'driverId'], as_index=False)['podium'].max())
pd.testing.assert_frame_equal(driver_race_labels, swapped_labels)
pd.testing.assert_frame_equal(pre_race_entries, build_pre_race_entries(tampered))
assert len(pre_race_entries) == len(results_dedup)
data['results'] = results_dedup
print('Passed: unique grain, unchanged podium set, independent input resolution, forbidden-outcome tampering and label-preserving outcome swaps.')


After deduplication, some early races still contain more than three rows with a recorded top-three `positionOrder`. The grain check establishes that these are different driver IDs, so the notebook retains their positive outcome labels as recorded. The CSVs alone do not explain the historical reason for every such row.

In [ ]:
podiums_per_race = data['results'].groupby('raceId')['positionOrder'].apply(lambda s: (s <= 3).sum())
podiums_per_race.value_counts().sort_index().rename_axis('podium rows in race').to_frame('races')

In [ ]:
# The other per-driver-per-race tables must satisfy the same grain
print('qualifying duplicates     :', data['qualifying'].duplicated(['raceId', 'driverId']).sum())
print('sprint_results duplicates :', data['sprint_results'].duplicated(['raceId', 'driverId']).sum())

#### Composite constructor audit

All individual keys passed above. This additional check compares constructor identity for the same driver and race across qualifying and results, then checks whether each race-constructor pair in results has matching constructor-result and constructor-standing records. This descriptive audit compares recorded result constructors. Modeling resolves constructor identity independently and leaves conflicting assignments missing. Missing standings remain missing; later work must use a left join and a documented historical lookup or missing indicator rather than dropping rows or filling all gaps with zero.


In [ ]:
# Composite agreement is separate from individual primary/foreign keys.
# The post-race descriptive constructor comes from results; modeling uses independently resolved pre_race_entries.constructorId. Qualifying.constructorId is
# independently audited; mismatches are never silently resolved by a join.
race_constructor = data['results'][['raceId', 'constructorId']].drop_duplicates()
quali_teams = data['qualifying'][['raceId', 'driverId', 'constructorId']].rename(
    columns={'constructorId': 'constructorId_qualifying'})
team_agreement = data['results'][['raceId', 'driverId', 'constructorId']].merge(
    quali_teams, on=['raceId', 'driverId'], how='inner', validate='one_to_one')
team_mismatch = team_agreement[
    team_agreement['constructorId'] != team_agreement['constructorId_qualifying']
].merge(data['races'][['raceId', 'year']], on='raceId')

join_audit = []
for name in ['constructor_results', 'constructor_standings']:
    covered = data[name][['raceId', 'constructorId']].drop_duplicates()
    check = race_constructor.merge(covered.assign(matched=True),
                                   on=['raceId', 'constructorId'], how='left', validate='one_to_one')
    missing = check[check['matched'].isna()].merge(
        data['races'][['raceId', 'year']], on='raceId')
    join_audit.append({'table': name, 'result race-team pairs': len(check),
                       'missing pairs all years': len(missing),
                       'missing pairs 2003-2024': int((missing['year'] >= 2003).sum())})
    # Keep the missing-pair indicator; a later join must be left-sided and
    # cannot turn every missing standing into zero without a temporal rule.
    if name == 'constructor_results': missing_constructor_results = missing
    else: missing_constructor_standings = missing

print('Results/qualifying constructor mismatches:', len(team_mismatch))
print(team_mismatch.groupby('year').size().rename('mismatches').to_frame())
print(pd.DataFrame(join_audit).to_string(index=False))
print('Missing constructor-results pairs by year:')
print(missing_constructor_results.groupby('year').size().rename('pairs').to_frame())
print('Missing constructor-standings pairs by year:')
print(missing_constructor_standings.groupby('year').size().rename('pairs').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
team_mismatch.groupby('year').size().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Different constructor IDs: results vs qualifying')
axes[0].set_ylabel('Driver-race rows')
audit_plot = pd.DataFrame({
    'constructor results': missing_constructor_results.groupby('year').size(),
    'constructor standings': missing_constructor_standings.groupby('year').size(),
}).fillna(0).astype(int)
audit_plot.plot.bar(ax=axes[1], color=['grey', 'steelblue'])
axes[1].set_title('Missing result race-constructor join pairs')
axes[1].set_ylabel('Pairs')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_4_composite_join_audit.png')
plt.show()


### 1.5 Inspect the data again (after cleaning)

We repeat the checks from 1.2 on the cleaned data to show the effect of each cleaning step.

#### 1.5.1 `info()` and `isnull().sum()` after cleaning

In [ ]:
data['results'].info()

In [ ]:
missing_compare = pd.DataFrame({
    'missing before (naive read_csv)': results_naive.isnull().sum(),
    'missing after cleaning': data['results'][results_naive.columns].isnull().sum(),
})
missing_compare[missing_compare.sum(axis=1) > 0]

**Observation:**
* The naive load reported **no** missing values; the cleaned table reveals the real ones.
* Columns are now numeric (`float64`/`int64`) instead of `object`, so they can be described, plotted and used by models.

In [ ]:
data['results'][['grid', 'grid_clean', 'positionOrder', 'points', 'laps', 'milliseconds', 'fastestLapTime_s']].describe().round(2)

#### 1.5.2 Missing values before and after interpreting source codes

The paired charts compare the **same raw rows** before and after treating `\N` and empty text as missing. In results, `grid = 0` is also treated as a missing numeric position. Each chart prints the exact percentages used for its bars, so the change can be checked without reading thousands of individual rows. Race-date sorting is retained for the later season-level check.

In [ ]:
race_dates = raw['races'][['raceId', 'date']].copy()
race_dates['raceId'] = race_dates['raceId'].astype(int)
race_dates['date'] = pd.to_datetime(race_dates['date'])

naive_dated = results_naive.merge(race_dates, on='raceId')
print('resultId order is chronological:', naive_dated.sort_values('resultId')['date'].is_monotonic_increasing)

results_naive_sorted = naive_dated.sort_values('date').reset_index(drop=True)
results_clean_sorted = data['results'].merge(data['races'][['raceId', 'date']], on='raceId') \
                                      .sort_values('date').reset_index(drop=True)

In [ ]:
cols = ['grid', 'position', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed']

def plot_missing_comparison(before, after, title, filename):
    """Compare the same source rows before/after recognizing missing-value codes."""
    assert len(before) == len(after) and list(before.columns) == list(after.columns)
    table = pd.DataFrame({
        'field': before.columns,
        'naive missing %': before.isna().mean().to_numpy() * 100,
        'semantic missing %': after.isna().mean().to_numpy() * 100,
    })
    table['newly visible %'] = table['semantic missing %'] - table['naive missing %']
    assert table['newly visible %'].ge(-1e-10).all()

    y = np.arange(len(table))
    fig, ax = plt.subplots(figsize=(10.8, max(3.5, 0.62 * len(table) + 1.3)))
    ax.set_axisbelow(True)
    ax.grid(axis='x', color='#DCE5EB', linewidth=0.8)
    ax.barh(y - 0.19, table['naive missing %'], height=0.34,
            color='#A9B6C2', label='Naive read')
    ax.barh(y + 0.19, table['semantic missing %'], height=0.34,
            color='#167A96', label='After recognizing codes')
    for row, yy in zip(table.itertuples(index=False), y):
        ax.text(row[1] + 1.1, yy - 0.19, f'{row[1]:.1f}%', va='center', fontsize=9, color='#334155')
        ax.text(row[2] + 1.1, yy + 0.19, f'{row[2]:.1f}%', va='center', fontsize=9, color='#0B5368')
    ax.set_yticks(y, table['field'])
    ax.invert_yaxis()
    ax.set_xlim(0, 111)
    ax.set_xticks([0, 20, 40, 60, 80, 100])
    ax.set_xlabel('Rows recorded as missing (%)')
    ax.set_title(title, loc='left', fontsize=12, pad=14)
    ax.legend(frameon=False, loc='lower right')
    ax.spines[['top', 'right', 'left']].set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{FIG_PATH}/{filename}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Same source rows on both sides: {len(before):,}')
    print(table.round(2).to_string(index=False))
    return table

# The same 26,759 raw result rows appear on both sides. The after view only
# recognizes the sentinel and grid-zero code; deduplication cannot confound
# this comparison.
results_semantic_missing = results_naive_sorted[cols].replace('\\N', np.nan).replace('', np.nan)
results_semantic_missing['grid'] = pd.to_numeric(
    results_semantic_missing['grid'], errors='coerce').replace(0, np.nan)
results_missing_compare = plot_missing_comparison(
    results_naive_sorted[cols], results_semantic_missing,
    'Results: missing values revealed by cleaning',
    '1_5_missing_results_comparison.png')

In [ ]:
qualifying_naive = pd.read_csv(f'{DATA_PATH}/qualifying.csv')
qcols = ['position', 'q1', 'q2', 'q3']
qualifying_naive = qualifying_naive.merge(race_dates, on='raceId').sort_values('date').reset_index(drop=True)
qualifying_clean_sorted = data['qualifying'].merge(data['races'][['raceId', 'date']], on='raceId') \
                                            .sort_values('date').reset_index(drop=True)
qualifying_missing_compare = plot_missing_comparison(
    qualifying_naive[qcols], qualifying_clean_sorted[qcols],
    'Qualifying: missing values revealed by cleaning',
    '1_5_missing_qualifying_comparison.png')

The paired charts show **how much** missingness becomes visible when source codes are interpreted. The next computations show how it varies by season and by race outcome. These group patterns help identify likely data mechanisms, while individual missing values may still have unknown causes.

In [ ]:
res_y = results_clean_sorted.merge(data['races'][['raceId', 'year']], on='raceId')
q_y = qualifying_clean_sorted.merge(data['races'][['raceId', 'year']], on='raceId')

missing_by_year = pd.concat([
    res_y.groupby('year')[['fastestLap', 'time']].apply(lambda d: d.isnull().mean() * 100),
    q_y.groupby('year')[['q2', 'q3']].apply(lambda d: d.isnull().mean() * 100),
], axis=1)

plt.figure(figsize=(12, 3.5))
sns.heatmap(missing_by_year.T, cmap='Reds', cbar_kws={'label': '% missing'}, xticklabels=5)
plt.title('% missing per season (blank = table has no rows that season)')
plt.xlabel('Season')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_missing_by_year.png')
plt.show()

first_recorded = missing_by_year.index[missing_by_year['fastestLap'] < 10].min()
print('fastestLap: first season with < 10 % missing:', first_recorded)
print('fastestLap % missing around it:', missing_by_year.loc[first_recorded - 2:first_recorded + 2, 'fastestLap'].round(1).to_dict())
print('q2 % missing, 2003-2008:', missing_by_year.loc[2003:2008, 'q2'].round(0).to_dict())

In [ ]:
# Outcome groups from the status text: 'Finished', lapped finishers ('+1 Lap', '+2 Laps', ...), and the rest
res_y = res_y.merge(data['status'], on='statusId')
res_y['outcome'] = 'did not finish'
res_y.loc[res_y['status'].str.startswith('+'), 'outcome'] = 'finished, lapped'
res_y.loc[res_y['status'] == 'Finished', 'outcome'] = 'finished, lead lap'

time_missing = res_y.groupby('outcome')['time'].apply(lambda t: t.isnull().mean() * 100)

plt.figure(figsize=(7, 3.5))
plt.bar(time_missing.index, time_missing.values, color='steelblue')
plt.title('% of rows with missing race time, by outcome')
plt.ylabel('% missing')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_time_missing_by_outcome.png')
plt.show()

time_missing.round(1)

**Patterns visible in the computed aggregates:**

* The share of missing fastest-lap fields changes sharply by season. This is consistent with a change in data coverage, but a missing value's individual cause is not established here.
* Recorded `time` availability differs by finishing-outcome group. The plot shows an association; absence of a time value does not by itself prove why it is absent.
* `q2` and `q3` have higher missing shares than `q1`, and their shares vary by season. The notebook preserves these as missing rather than imputing a lap time. Any interpretation as a qualifying-session outcome requires the corresponding qualifying format rules.

These are post-race or session-specific fields; their use as model features must be decided at the prediction cutoff.

#### 1.5.3 Effect on the target and on `grid`

In [ ]:
res_clean = data['results'].merge(data['races'][['raceId', 'year']], on='raceId')
res_clean['podium'] = (res_clean['positionOrder'] <= 3).astype(int)
confirmed_starters = res_clean[res_clean['entry_type'].eq('confirmed_start')]
season_clean = confirmed_starters.groupby('year')['podium'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(podium_rate_raw.index, podium_rate_raw.values, color='grey', linewidth=2, label='raw result rows')
axes[0].plot(season_clean.index, season_clean.values, color='steelblue', linewidth=2,
             label='deduplicated, confirmed starts')
axes[0].set_title('Podium rate per season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Podium rate (%)')
axes[0].legend()

bins = np.arange(0, 36)
axes[1].hist(res_year['grid'], bins=bins, color='grey', alpha=0.7, label='raw grid (0 = code)')
axes[1].hist(data['results']['grid_clean'].dropna(), bins=bins, color='steelblue', alpha=0.7, label='grid_clean')
axes[1].set_title('Grid distribution before vs. after')
axes[1].set_xlabel('Grid position')
axes[1].set_ylabel('Rows')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_target_and_grid.png')
plt.show()

print(f"Podium rate raw: {res_year['podium'].mean():.3f}")
print(f"Podium rate deduplicated: {res_clean['podium'].mean():.3f}")
print(f"Podium rate among confirmed starts: {confirmed_starters['podium'].mean():.3f}")
print('Unresolved start rows excluded from the confirmed-start curve:',
      res_clean['entry_type'].eq('unresolved_start').sum())

**Computed effects:** the cells above report raw result-row rate, rate after deduplication, and rate among confirmed starters separately. On this snapshot these are about 12.7%, 12.7%, and 14.1%. The last curve excludes unresolved start records and is therefore an evidence-based subset rather than every true starter. The grid histogram separately shows the effect of treating zero as a code instead of a numeric position.

#### 1.5.4 Distributions of the converted durations

These columns were text before cleaning and could not be plotted. Now we can check their ranges, using histograms for distributions and a boxplot to spot outliers, as in Lab 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for col, color in zip(['q1_s', 'q2_s', 'q3_s'], ['steelblue', 'orange', 'green']):
    axes[0].hist(data['qualifying'][col].dropna(), bins=60, range=(55, 130), alpha=0.5, color=color, label=col)
axes[0].set_title('Qualifying lap times')
axes[0].set_xlabel('Seconds')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(data['lap_times']['time_s'], bins=100, range=(50, 200), color='steelblue')
axes[1].set_title('Race lap times (50-200 s shown)')
axes[1].set_xlabel('Seconds')

sns.boxplot(x=data['pit_stops']['duration_s'], ax=axes[2], color='steelblue')
axes[2].set_xscale('log')
axes[2].set_title('Pit-stop durations (log scale)')
axes[2].set_xlabel('Seconds')

plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_durations.png')
plt.show()

data['pit_stops']['duration_s'].describe().round(1)
for col in ['q1_s', 'q2_s', 'q3_s']:
    observed = data['qualifying'][col].dropna()
    print(col, ':', int((~observed.between(55, 130)).sum()),
          'of', len(observed), 'observed values outside the displayed 55-130 s range')
msno.matrix(data['results'][['grid_clean', 'position', 'milliseconds', 'fastestLap', 'fastestLapTime_s']],
            figsize=(10, 3.5), sparkline=False, color=(0.16, 0.49, 0.56))
plt.title('Cleaned results: recorded and missing fields')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_missingno_matrix.png', dpi=150)
plt.show()


The next calculations quantify the share of Q1 time variation **between races** and **within races**. They do not attribute those components uniquely to circuit or driver. The long-stop calculation checks whether recorded stops over 100 seconds cluster within the same races and laps; clustering alone cannot establish the cause.

In [ ]:
q1_valid = data['qualifying'][['raceId', 'q1_s']].dropna().copy()
grand_mean = q1_valid['q1_s'].mean()
race_mean = q1_valid.groupby('raceId')['q1_s'].transform('mean')
total_ss = ((q1_valid['q1_s'] - grand_mean) ** 2).sum()
within_ss = ((q1_valid['q1_s'] - race_mean) ** 2).sum()
between_ss = ((race_mean - grand_mean) ** 2).sum()
assert np.isclose(total_ss, within_ss + between_ss)
between_share = 100 * between_ss / total_ss
within_share = 100 * within_ss / total_ss
print('Q1 observations:', len(q1_valid))
print(f'Between-race share of Q1 time variation: {between_share:.3f}%')
print(f'Within-race share of Q1 time variation: {within_share:.3f}%')

plt.figure(figsize=(6, 3.5))
plt.bar(['between races', 'within races'], [between_share, within_share],
        color=['steelblue', 'grey'])
plt.title('Exact decomposition of Q1 time variation')
plt.ylabel('Share of total sum of squares (%)')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_q1_variance_decomposition.png')
plt.show()

In [ ]:
long_stops = data['pit_stops'][data['pit_stops']['duration_s'] > 100]
per_race = long_stops.groupby('raceId').agg(long_stops=('driverId', 'count'),
                                            distinct_laps=('lap', 'nunique'))

print('Long stops (> 100 s):', len(long_stops), 'in', len(per_race), 'races')
print('Median long stops per race:', per_race['long_stops'].median(),
      '| median distinct laps they happen on:', per_race['distinct_laps'].median())

plt.figure(figsize=(7, 3.5))
plt.scatter(per_race['distinct_laps'], per_race['long_stops'], color='crimson')
plt.title('Long pit stops per race vs. number of distinct laps they occur on')
plt.xlabel('Distinct laps with a long stop')
plt.ylabel('Long stops in the race')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/1_5_long_pit_stops.png')
plt.show()

**Checks:** the Q1 time calculation above uses each observation in an exact sum-of-squares decomposition; the two components add to total variation. Between-race variation combines circuit, weather, era, qualifying format, and other race conditions. The long-stop plot reports clustering, which is evidence of a shared event but does not identify every stop's cause. Durations remain unchanged.

### 1.6 Summary of cleaning decisions

| Step | Applied rule | Programmatic evidence |
|---|---|---|
| Sentinels | Treat `\N` and empty text as missing | Full-table token counts and zero placeholders after replacement |
| Categories | Strip surrounding spaces and apply the declared country/nationality alias policy | Full-category checks, candidate plot, and before/after counts |
| Types | Convert numeric fields, dates, and durations | Parsed-value counts and millisecond reconciliation |
| Keys | Check all primary and foreign keys | Complete uniqueness, null, and orphan counts |
| Participation | Confirm starts from positive completed laps; keep every zero-lap row unresolved; treat grid zero as unavailable position | Full status-by-evidence table and grid-code crosstab |
| Grain | Aggregate driver-race podium labels; resolve current inputs without outcomes; retain a separate descriptive result representative | Label preservation, outcome-invariance and uniqueness assertions |

All cleaning runs on in-memory copies. The raw CSV files remain unchanged. The cleaned tables are stored in `data` for the following sections.


In [ ]:
for name in TABLE_NAMES:
    print(f'{name:<22} {data[name].shape}')

## 2. Data-engineering questions

Each answer uses multiple joined tables, aggregate evidence, plots and an explicit denominator. Positive completed laps confirm a start for these descriptive analyses. Every zero-lap entry remains unresolved; sensitivity analyses show how that uncertainty affects conclusions. These post-race fields are used to answer historical questions, not as current-race prediction inputs.

### 2.1 Q1 — Which circuits convert a front-row start into a podium most often, and does the ranking change after 2014?

**Tables joined:** `results` (grid, finishing position) + `races` (year, circuit) + `circuits` (circuit name) + `qualifying` (qualifying position).

**Definitions**
* **Front row** = the first two places on the starting grid, i.e. position 1 or 2.
* The spec asks us to distinguish **qualifying position** from **grid**, because grid penalties can move a driver after qualifying. So we compute front row **both ways**:
  * `front_row_grid`: `results.grid_clean` is 1 or 2, i.e. where the driver actually started;
  * `front_row_quali`: `qualifying.position` is 1 or 2.
* **Conversion rate** = share of front-row starters who finished on the podium (`positionOrder <= 3`).
* **Eras:** before 2014 vs. 2014 onward. The pre-2014 era starts at the first season with full qualifying coverage, computed in 1.2.6 as `full_from`. This way both front-row definitions are measured on exactly the same races.

In [ ]:
q1 = data['results'].merge(data['races'][['raceId', 'year', 'circuitId']], on='raceId')
q1 = q1.merge(data['circuits'][['circuitId', 'name']], on='circuitId')
q1 = q1.merge(data['qualifying'][['raceId', 'driverId', 'position']].rename(
    columns={'position': 'quali_position'}), on=['raceId', 'driverId'], how='left')
q1 = q1[q1['year'] >= full_from].copy()
q1['podium'] = q1['positionOrder'] <= 3

q1['grid_front_row_recorded'] = q1['grid_clean'].isin([1, 2])
q1['quali_front_row_recorded'] = q1['quali_position'].isin([1, 2])
q1['front_row_grid'] = q1['grid_front_row_recorded'] & q1['entry_type'].eq('confirmed_start')
q1['front_row_quali'] = q1['quali_front_row_recorded'] & q1['entry_type'].eq('confirmed_start')
q1['era'] = np.where(q1['year'] >= 2014, '2014 onward', 'before 2014')

quali_front_rows = q1[q1['quali_front_row_recorded']]
front_completeness = quali_front_rows.groupby('raceId').agg(
    front_row_drivers=('driverId', 'nunique'),
    front_row_positions=('quali_position', 'nunique'))
assert len(front_completeness) == q1['raceId'].nunique()
assert front_completeness[['front_row_drivers', 'front_row_positions']].eq(2).all().all()
print('Races with two distinct qualifying front-row drivers:', len(front_completeness))

print('Grain: duplicates of (raceId, driverId) =', q1.duplicated(['raceId', 'driverId']).sum())
print('Rows and seasons:', len(q1), q1['year'].min(), '-', q1['year'].max())
print('Rows without qualifying record:', q1['quali_position'].isna().sum())
print('Rows without numeric grid position:', q1['grid_clean'].isna().sum())
print('Front-row candidates with unresolved participation, grid:',
      int((q1['grid_front_row_recorded'] & q1['entry_type'].eq('unresolved_start')).sum()))
print('Front-row candidates with unresolved participation, qualifying:',
      int((q1['quali_front_row_recorded'] & q1['entry_type'].eq('unresolved_start')).sum()))

**Missingness and participation:** all analyzed races have two distinct recorded qualifying front-row positions, so drivers without a qualifying row cannot occupy P1/P2 in this snapshot. A missing numeric grid position cannot be P1/P2 by the grid definition. Front-row records with unresolved participation are excluded from the confirmed-start point rates below and included in rate bounds. The next check compares the two front-row definitions among confirmed starters.

In [ ]:
q1_confirmed = q1.loc[q1['entry_type'].eq('confirmed_start')].copy()
agreement = pd.pivot_table(q1_confirmed, index='front_row_quali', columns='front_row_grid',
                           values='driverId', aggfunc='count', fill_value=0)
assert int(agreement.to_numpy().sum()) == len(q1_confirmed)
q1_agreement_audit = (q1_confirmed.groupby(['front_row_quali', 'front_row_grid']).size().reset_index(name='entries'))
print('Agreement population: confirmed starters', len(q1_confirmed), '| excluded unresolved entries', len(q1) - len(q1_confirmed))
sns.heatmap(agreement, annot=True, fmt='d', cmap='Blues')
plt.title('Confirmed front-row starts: qualifying vs recorded grid')
plt.xlabel('Front row by grid')
plt.ylabel('Front row by qualifying')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_1_quali_vs_grid.png')
plt.show()

uncertain_front_row = q1[q1['entry_type'].eq('unresolved_start')].groupby('era').agg(
    recorded_grid=('grid_front_row_recorded', 'sum'),
    recorded_qualifying=('quali_front_row_recorded', 'sum'))
print('Unresolved starts with a recorded front-row position:')
print(uncertain_front_row.to_string())
uncertain_front_row.plot(kind='bar', figsize=(7, 3.5), color=['grey', 'steelblue'])
plt.title('Recorded front-row positions with unresolved starts')
plt.xlabel('Era')
plt.ylabel('Driver-race rows')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_1_front_row_uncertainty.png')
plt.show()


The heatmap compares qualifying and stored grid positions **among confirmed starters**. The table and plot separately show records whose front-row position is known but whose start cannot be established from these CSVs. Such records are not silently counted as starts or treated as non-starts.

In [ ]:
starts_per_circuit = q1[q1['front_row_grid']].groupby(['name', 'era'])['driverId'].count().reset_index(name='front_row_starts')

plt.figure(figsize=(8, 3.5))
plt.hist(starts_per_circuit['front_row_starts'], bins=range(0, 30, 2), color='grey')
plt.axvline(10, color='crimson', linestyle='--')
plt.title('Front-row starts per circuit per era (grid definition)')
plt.xlabel('Front-row starts')
plt.ylabel('Circuit-era pairs')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_1_sample_size.png')
plt.show()

**Choice:** a circuit is included only if it has **at least 10 front-row starts in each era**, i.e. at least 5 races. At that size, one podium moves the rate by at most 10 points. Circuits visited in only one era cannot show a change between eras, so they are excluded too.

In [ ]:
MIN_STARTS = 10

def conversion_table(front_row_column):
    confirmed = q1[q1[front_row_column]]
    rate = confirmed.groupby(['name', 'era'])['podium'].mean().unstack() * 100
    count = confirmed.groupby(['name', 'era'])['podium'].count().unstack()
    keep = (count['before 2014'] >= MIN_STARTS) & (count['2014 onward'] >= MIN_STARTS)
    return rate[keep], count[keep]

rate_grid, count_grid = conversion_table('front_row_grid')
rate_quali, count_quali = conversion_table('front_row_quali')
print('Circuits meeting the confirmed-start sample-size rule:', len(rate_grid))

def front_row_bounds(recorded_column):
    candidate = q1[q1[recorded_column]].copy()
    candidate['confirmed'] = candidate['entry_type'].eq('confirmed_start')
    assert not (candidate['podium'] & ~candidate['confirmed']).any()
    totals = candidate.groupby(['name', 'era']).agg(
        confirmed_starts=('confirmed', 'sum'),
        unresolved_starts=('confirmed', lambda x: (~x).sum()),
        podiums=('podium', 'sum'))
    totals['lower %'] = 100 * totals['podiums'] / (
        totals['confirmed_starts'] + totals['unresolved_starts'])
    totals['upper %'] = 100 * totals['podiums'] / totals['confirmed_starts'].replace(0, np.nan)
    return totals

grid_front_row_bounds = front_row_bounds('grid_front_row_recorded')
quali_front_row_bounds = front_row_bounds('quali_front_row_recorded')
print('Grid front-row uncertainty for eligible circuits:')
print(grid_front_row_bounds.loc[grid_front_row_bounds.index.get_level_values('name').isin(rate_grid.index)]
      .round(2).to_string())

In [ ]:
# Grouped bar chart: one pair of bars per circuit (grid definition), sorted by the 2014-onward rate
plot_df = rate_grid.sort_values('2014 onward')

plot_df[['before 2014', '2014 onward']].plot(kind='barh', figsize=(9, 7), color=['grey', 'steelblue'])
plt.title('Front-row (grid) to podium conversion rate by circuit')
plt.xlabel('Podium rate of front-row starters (%)')
plt.ylabel('')
plt.legend(title='Era')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_1_conversion_by_circuit.png')
plt.show()

#### Does the ranking change after 2014?

Each circuit is ranked within each era: rank 1 means the highest conversion rate. The rank change is shown for both definitions of front row.

In [ ]:
ranking = pd.DataFrame({
    'rate before 2014 (grid)': rate_grid['before 2014'].round(1),
    'rate 2014 onward (grid)': rate_grid['2014 onward'].round(1),
    'rank before 2014 (grid)': rate_grid['before 2014'].rank(ascending=False, method='min'),
    'rank 2014 onward (grid)': rate_grid['2014 onward'].rank(ascending=False, method='min'),
    'rate before 2014 (quali)': rate_quali['before 2014'].round(1),
    'rate 2014 onward (quali)': rate_quali['2014 onward'].round(1),
    'front-row starts before / after (grid)': count_grid['before 2014'].astype(int).astype(str) + ' / '
                                              + count_grid['2014 onward'].astype(int).astype(str),
})
ranking['rank change (grid)'] = ranking['rank before 2014 (grid)'] - ranking['rank 2014 onward (grid)']
ranking = ranking.sort_values('rank 2014 onward (grid)')
ranking

In [ ]:
# Positive = the circuit moved UP the ranking after 2014, negative = it moved down
rank_change = ranking['rank change (grid)'].sort_values()
colors = ['crimson' if change < 0 else 'steelblue' for change in rank_change]

plt.figure(figsize=(8, 5.5))
plt.barh(rank_change.index, rank_change.values, color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Change in conversion-rate rank after 2014 (grid definition)')
plt.xlabel('Places moved (positive = moved up)')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_1_rank_change.png')
plt.show()

top5_before = set(ranking.sort_values('rank before 2014 (grid)').index[:5])
top5_after = set(ranking.sort_values('rank 2014 onward (grid)').index[:5])
print('Top 5 before 2014 :', sorted(top5_before))
print('Top 5 2014 onward :', sorted(top5_after))
print('Circuits in both top 5 lists:', len(top5_before & top5_after))

overall = q1[q1['front_row_grid']].groupby('era')['podium'].mean() * 100
print('Overall front-row conversion (grid) per era:', overall.round(1).to_dict())

In [ ]:
post_bounds = grid_front_row_bounds.xs('2014 onward', level='era').loc[rate_grid.index]
leader_confirmed = rate_grid['2014 onward'].idxmax()
leader_all = post_bounds['lower %'].idxmax()
q1_answer = (f'Among {len(rate_grid)} eligible circuits with at least {MIN_STARTS} confirmed front-row starts in each era, '
             f'{leader_confirmed} leads after 2014 when zero-lap entries are excluded '
             f'({rate_grid.loc[leader_confirmed, "2014 onward"]:.1f}%). '
             f'When every unresolved front-row entry is counted as a start, {leader_all} leads '
             f'({post_bounds.loc[leader_all, "lower %"]:.1f}%). '
             'The leader therefore depends on zero-lap participation. '
             f'{len(top5_before & top5_after)} circuits appear in both era top-five lists. '
             'Qualifying position and recorded grid are separate definitions; the recorded qualifying front row is complete in the analyzed window.')
display(Markdown('**Answer (Q1):** ' + q1_answer))


### 2.2 Q2 — Do drivers podium more often at a circuit in their home country, after controlling for starting position?

**Tables joined:** `results` (grid, finishing position) + `races` (circuit) + `circuits` (country) + `drivers` (nationality).

#### Matching a nationality to a country

`drivers.nationality` holds a nationality word (e.g. `British`), while `circuits.country` holds a country name (e.g. `UK`). The dataset has no table linking the two, so we write the mapping **country → nationality word** for every circuit country. Two code checks verify it:
1. every circuit country has a mapping;
2. we print which mapped nationality words actually occur in `drivers`. A word that never occurs simply means no driver from that country ever raced, so no home races are possible for it.

In [ ]:
country_to_nationality = {
    'Argentina': 'Argentine', 'Australia': 'Australian', 'Austria': 'Austrian', 'Azerbaijan': 'Azerbaijani',
    'Bahrain': 'Bahraini', 'Belgium': 'Belgian', 'Brazil': 'Brazilian', 'Canada': 'Canadian',
    'China': 'Chinese', 'France': 'French', 'Germany': 'German', 'Hungary': 'Hungarian',
    'India': 'Indian', 'Italy': 'Italian', 'Japan': 'Japanese', 'Korea': 'Korean',
    'Malaysia': 'Malaysian', 'Mexico': 'Mexican', 'Monaco': 'Monegasque', 'Morocco': 'Moroccan',
    'Netherlands': 'Dutch', 'Portugal': 'Portuguese', 'Qatar': 'Qatari', 'Russia': 'Russian',
    'Saudi Arabia': 'Saudi', 'Singapore': 'Singaporean', 'South Africa': 'South African', 'Spain': 'Spanish',
    'Sweden': 'Swedish', 'Switzerland': 'Swiss', 'Turkey': 'Turkish', 'UAE': 'Emirati',
    'UK': 'British', 'USA': 'American',
}

circuit_countries = set(data['circuits']['country'])
driver_nationalities = set(data['drivers']['nationality'])

print('Circuit countries without a mapping:', circuit_countries - set(country_to_nationality))
print('Mapped nationality words that occur in drivers:',
      sorted(set(country_to_nationality.values()) & driver_nationalities))
print('Mapped nationality words with no driver:',
      sorted(set(country_to_nationality.values()) - driver_nationalities))

In [ ]:
q2 = data['results'].merge(data['races'][['raceId', 'year', 'circuitId']], on='raceId')
q2 = q2.merge(data['circuits'][['circuitId', 'country']], on='circuitId')
q2 = q2.merge(data['drivers'][['driverId', 'nationality']], on='driverId')

print('Missing nationality:', q2['nationality'].isna().sum(),
      '| missing country:', q2['country'].isna().sum())
hyphenated = q2['nationality'].str.contains('-')
print('Rows with a hyphenated nationality (treated as not home):', hyphenated.sum())

q2['home'] = q2['nationality'] == q2['country'].map(country_to_nationality)
q2['podium'] = q2['positionOrder'] <= 3
before = len(q2)
q2_population = q2.copy()
print('Rows by participation evidence before filtering:')
print(q2['entry_type'].value_counts().to_string())
q2 = q2[q2['entry_type'].eq('confirmed_start') & q2['grid_clean'].notna()].copy()
print(f'Confirmed starts with a grid position: {len(q2)} of {before}')
print('Home rows:', q2['home'].sum(), '| away rows:', (~q2['home']).sum())

**Missingness and denominator:** nationality and circuit country are complete in the printed checks. The comparison uses confirmed starts with a positive numeric grid position, because grid buckets require one. Records with unresolved participation or a nonnumeric grid are counted separately rather than silently treated as starters. Hyphenated nationality labels follow the stated mapping policy and their count is printed.

#### Why controlling for grid is needed

Home and away drivers can have different recorded grid distributions. The next aggregate check reports those distributions before comparing podium rates within grid buckets.

In [ ]:
print('Average grid position:', q2.groupby('home')['grid_clean'].mean().round(2).rename({False: 'away', True: 'home'}).to_dict())
print('Raw podium rate (%):  ', (q2.groupby('home')['podium'].mean() * 100).round(1).rename({False: 'away', True: 'home'}).to_dict())

Home and away drivers start from different average grid positions, so the raw podium rates are **not** a fair comparison.

#### Grid buckets

To compare drivers who started from similar positions, grid is cut into buckets with `pd.cut`, as in Lab 1. The bucket edges follow the podium-by-grid plot in 1.2.5, which showed the rate falling steeply over the first few places and then flattening:
* `1-3`: the podium positions themselves;
* `4-6` and `7-10`: still a real podium chance;
* `11+`: podium chance close to zero.

Each bucket must contain enough **home** rows for its rate to be meaningful; the counts are shown below.

In [ ]:
q2['grid_bucket'] = pd.cut(q2['grid_clean'], bins=[0, 3, 6, 10, 40], labels=['1-3', '4-6', '7-10', '11+'])

rate_by_bucket = pd.pivot_table(q2, index='grid_bucket', columns='home', values='podium',
                                aggfunc='mean', observed=True) * 100
count_by_bucket = pd.pivot_table(q2, index='grid_bucket', columns='home', values='podium',
                                 aggfunc='count', observed=True)
rate_by_bucket.columns = ['away', 'home']
count_by_bucket.columns = ['away rows', 'home rows']

q2_table = pd.concat([rate_by_bucket.round(1), count_by_bucket], axis=1)
q2_table['home - away (points)'] = (q2_table['home'] - q2_table['away']).round(1)
q2_table

In [ ]:
rate_by_bucket.plot(kind='bar', figsize=(8, 4), color=['grey', 'steelblue'])
plt.title('Podium rate by starting-grid bucket: home vs. away')
plt.xlabel('Grid bucket')
plt.ylabel('Podium rate (%)')
plt.xticks(rotation=0)
plt.legend(title='')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_2_home_vs_away.png')
plt.show()

In [ ]:
# Calculate with full precision; round only the displayed result.
weights = count_by_bucket['home rows']
controlled_difference = ((rate_by_bucket['home'] - rate_by_bucket['away']) * weights).sum() / weights.sum()
raw_difference = q2[q2['home']]['podium'].mean() * 100 - q2[~q2['home']]['podium'].mean() * 100

possible_starters = q2_population[q2_population['grid_clean'].notna()].copy()
possible_starters['grid_bucket'] = pd.cut(
    possible_starters['grid_clean'], bins=[0, 3, 6, 10, 40], labels=['1-3', '4-6', '7-10', '11+'])
possible_rates = pd.pivot_table(possible_starters, index='grid_bucket', columns='home',
                                values='podium', aggfunc='mean', observed=True) * 100
possible_counts = pd.pivot_table(possible_starters, index='grid_bucket', columns='home',
                                 values='podium', aggfunc='count', observed=True)
possible_difference = ((possible_rates[True] - possible_rates[False]) *
                       possible_counts[True]).sum() / possible_counts[True].sum()

print(f'Raw difference (home - away)        : {raw_difference:+.3f} points')
print(f'Grid-bucket weighted difference     : {controlled_difference:+.3f} points')
print('Sensitivity including unresolved starts with a numeric grid position:')
print(f'  rows: {len(possible_starters)}; grid-bucket difference: {possible_difference:+.3f} points')

plt.figure(figsize=(6, 3.5))
plt.bar(['raw', 'within grid buckets'], [raw_difference, controlled_difference],
        color=['grey', 'steelblue'])
plt.axhline(0, color='black', linewidth=1)
plt.title('Home minus away podium rate, confirmed starters')
plt.ylabel('Percentage points')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_2_raw_vs_controlled.png')
plt.show()

In [ ]:
q2_answer = (f'Among {len(q2):,} positive-lap starters with a numeric grid, including '
             f'{int(q2["home"].sum()):,} home entries, the raw home-minus-away podium difference is '
             f'{raw_difference:+.3f} percentage points. Within-grid-bucket weighting gives '
             f'{controlled_difference:+.3f} points. Including unresolved zero-lap entries gives '
             f'{possible_difference:+.3f} points. These are associations within broad grid buckets, not causal effects.')
display(Markdown('**Answer (Q2):** ' + q2_answer))


## 2.3 Q3: mechanical-retirement rates

Group status labels only as completed/mechanical/non-mechanical/unknown for this
descriptive question; no engine/component classifications enter the model. Positive
laps confirm participation conservatively. A mechanical status with full winner-distance
laps and a recorded race time creates contradictory completion evidence, so its cause
is unresolved and excluded from the confirmed mechanical numerator. Raw data stay
unchanged. Show **all constructors** first, then a separately labeled >=40-start view
and matched-ID sensitivity. Publish denominators, uncertain causes and zero-lap bounds.


In [ ]:
MECHANICAL_STATUS = {
    'Engine', 'Gearbox', 'Transmission', 'Clutch', 'Hydraulics', 'Electrical',
    'Radiator', 'Suspension', 'Brakes', 'Differential', 'Overheating', 'Mechanical',
    'Driveshaft', 'Fuel pressure', 'Water pressure', 'Throttle', 'Steering',
    'Electronics', 'Heat shield fire', 'Exhaust', 'Oil leak', 'Water leak',
    'Fuel pump', 'Oil pressure', 'Engine fire', 'Engine misfire', 'Pneumatics',
    'Wheel bearing', 'Fuel system', 'Oil line', 'Launch control', 'Power loss',
    'Drivetrain', 'Ignition', 'Battery', 'Halfshaft', 'Crankshaft', 'Alternator',
    'Oil pump', 'Fuel leak', 'Injection', 'Distributor', 'Turbo', 'CV joint',
    'Water pump', 'Spark plugs', 'Fuel pipe', 'Oil pipe', 'Water pipe', 'Magneto',
    'Supercharger', 'Power Unit', 'ERS', 'Cooling system',
}
NON_MECHANICAL_STATUS = {
    'Disqualified', 'Accident', 'Collision', 'Spun off', 'Refuelling', 'Withdrew',
    'Out of fuel', 'Physical', 'Fuel rig', 'Injured', '107% Rule', 'Safety',
    'Did not qualify', 'Injury', 'Stalled', 'Safety concerns', 'Not restarted',
    'Underweight', 'Excluded', 'Did not prequalify', 'Driver unwell',
    'Fatal accident', 'Eye injury', 'Collision damage', 'Illness',
}
def retirement_group(label):
    if label == 'Finished' or (isinstance(label, str) and label.startswith('+') and 'Lap' in label):
        return 'completed'
    if label in MECHANICAL_STATUS:
        return 'mechanical'
    if label in NON_MECHANICAL_STATUS:
        return 'non-mechanical'
    return 'unknown'

status_policy = data['status'].copy()
status_policy['cause_group'] = status_policy['status'].map(retirement_group)
display(status_policy.groupby('cause_group')['status'].apply(lambda s: ', '.join(s)).to_frame('labels'))
q3 = (data['results'].merge(data['races'][['raceId', 'year']], on='raceId')
      .merge(data['constructors'][['constructorId', 'name']], on='constructorId')
      .merge(status_policy, on='statusId'))
q3 = q3[q3['year'].between(2014, 2024)].copy()
q3['period'] = '2014-2021'
q3.loc[q3['year'].ge(2022), 'period'] = '2022-2024'
q3['race_max_laps'] = q3.groupby('raceId')['laps'].transform('max')
q3['completion_conflict'] = (q3['cause_group'].eq('mechanical') & q3['laps'].gt(0)
    & q3['laps'].ge(q3['race_max_laps']) & q3['milliseconds'].notna())
q3_completion_conflicts = q3.loc[q3['completion_conflict'],
    ['resultId', 'raceId', 'driverId', 'name', 'year', 'positionOrder', 'laps', 'race_max_laps', 'time', 'status']].copy()
# Contradictory completion/cause evidence is unresolved, not a confirmed retirement.
q3.loc[q3['completion_conflict'], 'cause_group'] = 'unknown'
print('Mechanical-label/full-distance contradictions (raw CSV unchanged):')
display(q3_completion_conflicts)
q3['confirmed'] = q3['laps'].gt(0)
q3['mechanical_confirmed'] = q3['confirmed'] & q3['cause_group'].eq('mechanical')
q3['unknown_confirmed'] = q3['confirmed'] & q3['cause_group'].eq('unknown')
q3['unresolved'] = ~q3['confirmed']
q3['unresolved_mechanical'] = q3['unresolved'] & q3['cause_group'].eq('mechanical')
q3['unresolved_potential_failure'] = q3['unresolved'] & q3['cause_group'].isin(['mechanical', 'unknown'])
q3_table = q3.groupby(['constructorId', 'name', 'period']).agg(
    confirmed_starts=('confirmed', 'sum'), mechanical=('mechanical_confirmed', 'sum'),
    unknown_causes=('unknown_confirmed', 'sum'), zero_lap_entries=('unresolved', 'sum'),
    zero_lap_mechanical=('unresolved_mechanical', 'sum'),
    zero_lap_potential_failure=('unresolved_potential_failure', 'sum')).reset_index()
q3_table['mechanical rate %'] = 100 * q3_table['mechanical'] / q3_table['confirmed_starts']
q3_table['possible-start sensitivity %'] = 100 * (
    q3_table['mechanical'] + q3_table['zero_lap_mechanical']) / (
    q3_table['confirmed_starts'] + q3_table['zero_lap_entries'])
q3_table['lower %'] = 100 * q3_table['mechanical'] / (
    q3_table['confirmed_starts'] + q3_table['zero_lap_entries'])
q3_table['upper %'] = 100 * (
    q3_table['mechanical'] + q3_table['unknown_causes'] + q3_table['zero_lap_potential_failure']) / (
    q3_table['confirmed_starts'] + q3_table['zero_lap_potential_failure'])
MIN_Q3_STARTS = 40
q3_eligible = q3_table[q3_table['confirmed_starts'].ge(MIN_Q3_STARTS)].copy()
q3_excluded = q3_table[q3_table['confirmed_starts'].lt(MIN_Q3_STARTS)].copy()
# Missing periods are absent observations, not zero-retirement seasons.
# The pivot's zero fill is used only to check the secondary sample threshold.
eligibility = q3_table.pivot(index='constructorId', columns='period', values='confirmed_starts').fillna(0)
matched_q3_ids = eligibility.index[eligibility.ge(MIN_Q3_STARTS).all(axis=1)]
q3_matched = q3_eligible[q3_eligible['constructorId'].isin(matched_q3_ids)].copy()
q3_table['low_sample'] = q3_table['confirmed_starts'].lt(MIN_Q3_STARTS)
print('Unrestricted primary rates, with low-sample flag:')
display(q3_table.sort_values(['period', 'mechanical rate %'], ascending=[True, False]).round(2))
print('Secondary sample-filtered rates:')
display(q3_eligible.sort_values(['period', 'mechanical rate %'], ascending=[True, False]).round(2))
print('Constructor-period exclusions from the secondary >=40-start rankings (<40 confirmed starts in that period):')
display(q3_excluded[['name', 'period', 'confirmed_starts']].sort_values(['name', 'period']))
print('Constructors meeting the secondary size rule:')
print(q3_eligible.groupby('period')['constructorId'].nunique().to_string())
print('Secondary comparison:', len(matched_q3_ids), 'constructor IDs eligible in both periods')


In [ ]:
largest_period = q3_table.groupby('period').size().max()
fig, axes = plt.subplots(1, 2, figsize=(16, max(6, 0.46 * largest_period + 1)))
q3_leaders = []
q3_max_rate = q3_table[['mechanical rate %', 'possible-start sensitivity %']].max().max()
for ax, period in zip(axes, ['2014-2021', '2022-2024']):
    chart = q3_table[q3_table['period'].eq(period)].sort_values('mechanical rate %')
    ax.barh(chart['name'], chart['mechanical rate %'], color='#287D8E', label='Explicit mechanical labels')
    ax.scatter(chart['possible-start sensitivity %'], chart['name'], color='#B95D4C',
               marker='D', label='Including all zero-lap entries')
    ax.set_title(period, fontsize=18)
    ax.set_xlabel('Mechanical retirements / starts (%)', fontsize=15)
    ax.tick_params(axis='y', labelsize=15)
    ax.set_xlim(0, q3_max_rate * 1.15)
    ax.legend(fontsize=10)
    leader = chart.iloc[-1]
    q3_leaders.append(f'{period}: {leader["name"]} ({leader["mechanical rate %"]:.2f}%)')
fig.suptitle('Primary rankings: all constructors, denominators and low-sample flags in table', fontsize=17)
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/2_3_mechanical_retirements.png', dpi=150)
plt.show()
q3_filtered_leaders = []
for period, group in q3_eligible.groupby('period'):
    leader = group.loc[group['mechanical rate %'].idxmax()]
    q3_filtered_leaders.append(f'{period}: {leader["name"]} ({leader["mechanical rate %"]:.2f}%)')
q3_answer = ('Among all recorded constructor-periods with confirmed starts, the highest status-based '
    'mechanical-retirement rates are ' + '; '.join(q3_leaders) + '. '
    'The secondary >=40-confirmed-start comparison instead gives ' + '; '.join(q3_filtered_leaders) + '. '
    'Low-sample teams remain in the primary answer. Contradictory full-distance mechanical labels are unknown, '
    'not confirmed retirements. Zero-lap participation and cause ambiguity limit interpretation; '
    'bounds are policy-conditional, not confidence intervals. Constructor IDs remain separate across rebrands.')
display(Markdown('**Answer (Q3):** ' + q3_answer))

matched_rates = q3_matched.pivot(index='name', columns='period', values='mechanical rate %')
display(Markdown('**Secondary analysis: the same constructor IDs in both periods.**'))
display(q3_matched.sort_values(['name', 'period']).round(2))
matched_rates.sort_values('2014-2021').plot.barh(figsize=(10, 6.5), color=['#287D8E', '#B95D4C'])
plt.title('Secondary comparison: constructor IDs with at least 40 starts in both periods')
plt.xlabel('Status-based mechanical-retirement rate (%)')
plt.ylabel('')
plt.legend(title='Period')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/2_3_matched_constructor_comparison.png', dpi=150)
plt.show()
matched_leaders = []
for period, group in q3_matched.groupby('period'):
    leader = group.loc[group['mechanical rate %'].idxmax()]
    matched_leaders.append(f'{period}: {leader["name"]} ({leader["mechanical rate %"]:.2f}%)')
q3_matched_answer = (f'The secondary comparison contains {len(matched_q3_ids)} constructor IDs with at least '
    '40 confirmed starts in both periods. Within this restricted common set, leaders are ' +
    '; '.join(matched_leaders) + '. This answers a same-ID comparison and does not replace the independent period rankings.')
display(Markdown(q3_matched_answer))


## 3. Pre-race features, selection and chronological preparation

Retain every recorded driver-race entry from 1950-2024, including grid-zero and
zero-lap entries. Modeling joins independently resolved pre-race inputs to a
separately aggregated podium label. Ambiguous historical grid/constructor
assignments remain missing. Outcomes never choose current prediction inputs.

Grid is an allowed retrospective pre-race proxy; qualifying is retained for Q1
and omitted from predictors under confirmed TA guidance. Zero is a known code,
not ordinal P0. No current-race status, laps or mechanical component category is
a predictor. Q3's status grouping serves only its historical descriptive question.


In [ ]:
race_context = data['races'][['raceId', 'year', 'round', 'date', 'circuitId']].copy()
all_entries = pre_race_entries.merge(driver_race_labels,
    on=['raceId', 'driverId'], how='left', validate='one_to_one')
all_entries = all_entries.merge(race_context, on='raceId', how='left', validate='many_to_one')
assert all_entries['podium'].notna().all()
modeling = all_entries[all_entries['year'].between(1950, 2024)].copy()
expected_rows = len(modeling)
modeling = modeling.merge(data['drivers'][['driverId', 'dob']], on='driverId', how='left', validate='many_to_one')
assert len(modeling) == expected_rows and not modeling.duplicated(['raceId', 'driverId']).any()
modeling = modeling.sort_values(['date', 'raceId', 'driverId']).reset_index(drop=True)
modeling['grid_raw'] = modeling['grid']
modeling['grid_is_zero'] = modeling['grid'].eq(0).astype(int)
modeling['grid_position'] = modeling['grid'].where(modeling['grid'].gt(0))
modeling['grid_zero_reason_pre_race'] = pd.NA
modeling['driver_age_years'] = (modeling['date'] - modeling['dob']).dt.days / 365.25
print('Recorded entries:', len(modeling), '| races:', modeling['raceId'].nunique())
print('Grid-zero entries:', int(modeling['grid_is_zero'].sum()))


### 3.1 Earlier-race form and standings

Driver form uses the previous five recorded entries; constructor form first aggregates teammates within a race and then uses the previous five races. All sources must have a date strictly earlier than the target date, including the historical case of two races on the same day. Prior standings use the latest earlier snapshot in the same season. Current-race positions, points, status, laps and pit stops are forbidden inputs.

In [ ]:
def add_past_form(table, entity, outcome, count_name, rate_name, source_name):
    result = table.sort_values(['date', 'raceId']).copy()
    for _, group in result.groupby(entity, sort=False):
        dates = group['date'].to_numpy()
        # left boundary excludes every race on the target date.
        boundary = np.searchsorted(dates, dates, side='left')
        cumulative = np.r_[0.0, group[outcome].to_numpy().cumsum()]
        length = np.minimum(boundary, 5)
        form = np.divide(cumulative[boundary] - cumulative[np.maximum(0, boundary - 5)],
                         length, out=np.full(len(group), np.nan), where=length > 0)
        result.loc[group.index, count_name] = boundary
        result.loc[group.index, rate_name] = form
        result.loc[group.index, source_name] = [dates[b - 1] if b else pd.NaT for b in boundary]
    result[source_name] = pd.to_datetime(result[source_name])
    return result

history = add_past_form(all_entries, 'driverId', 'podium', 'driver_prior_entries',
                        'driver_prior5_podium_rate', 'driver_form_source_date')
# Historical outcomes can be attributed to the recorded car only after that race.
constructor_outcomes = (results_all_records.assign(podium=results_all_records['positionOrder'].le(3).astype(int))
    .groupby(['raceId', 'constructorId', 'driverId'], as_index=False)['podium'].max()
    .merge(race_context[['raceId', 'date']], on='raceId', validate='many_to_one'))
team_races = constructor_outcomes.groupby(['raceId', 'constructorId', 'date'], as_index=False).agg(
    race_podium_fraction=('podium', 'mean'))
team_races = add_past_form(team_races, 'constructorId', 'race_podium_fraction',
                           'constructor_prior_races', 'constructor_prior5_podium_rate',
                           'constructor_form_source_date')
modeling = modeling.merge(history[['raceId', 'driverId', 'driver_prior_entries',
    'driver_prior5_podium_rate', 'driver_form_source_date']], on=['raceId', 'driverId'], how='left')
modeling = modeling.merge(team_races[['raceId', 'constructorId', 'constructor_prior_races',
    'constructor_prior5_podium_rate', 'constructor_form_source_date']], on=['raceId', 'constructorId'], how='left')


In [ ]:
def prior_standings(model, standings, entity_col, prefix):
    snapshots = standings[['raceId', entity_col, 'points', 'position']].copy()
    assert not snapshots.duplicated(['raceId', entity_col]).any()
    snapshots = snapshots.merge(
        race_context[['raceId', 'year', 'date']], on='raceId', how='left', validate='many_to_one')
    assert snapshots['date'].notna().all()
    snapshots = snapshots.rename(columns={
        'raceId': prefix + '_source_raceId', 'date': prefix + '_source_date',
        'points': prefix + '_prior_points', 'position': prefix + '_prior_position'})
    base = model[['raceId', entity_col, 'year', 'date']].drop_duplicates(['raceId', entity_col])
    known = base[base[entity_col].notna()].copy()
    known[entity_col] = known[entity_col].astype(snapshots[entity_col].dtype)
    lookup_known = pd.merge_asof(
        known.sort_values('date'),
        snapshots.sort_values(prefix + '_source_date'),
        left_on='date', right_on=prefix + '_source_date', by=['year', entity_col],
        direction='backward', allow_exact_matches=False)
    columns = ['raceId', entity_col, prefix + '_prior_points', prefix + '_prior_position', prefix + '_source_date']
    lookup = base.merge(lookup_known[columns], on=['raceId', entity_col], how='left', validate='one_to_one')
    assert len(lookup) == len(model[['raceId', entity_col]].drop_duplicates())
    assert (lookup.loc[lookup[prefix + '_source_date'].notna(), prefix + '_source_date']
            < lookup.loc[lookup[prefix + '_source_date'].notna(), 'date']).all()
    return lookup[['raceId', entity_col, prefix + '_prior_points',
                   prefix + '_prior_position', prefix + '_source_date']]

driver_prior = prior_standings(modeling, data['driver_standings'], 'driverId', 'driver_standing')
team_prior = prior_standings(modeling, data['constructor_standings'], 'constructorId', 'constructor_standing')
modeling = modeling.merge(driver_prior, on=['raceId', 'driverId'], how='left', validate='one_to_one')
assert len(modeling) == expected_rows
modeling = modeling.merge(team_prior, on=['raceId', 'constructorId'], how='left', validate='many_to_one')
assert len(modeling) == expected_rows
for source in ['driver_standing_source_date', 'constructor_standing_source_date']:
    assert (modeling.loc[modeling[source].notna(), source]
            < modeling.loc[modeling[source].notna(), 'date']).all()
assert not modeling.duplicated(['raceId', 'driverId']).any()
print('Prior-standings coverage (%):',
      (100 * modeling[['driver_standing_prior_points', 'constructor_standing_prior_points']]
       .notna().mean()).round(1).to_dict())


### 3.2 Split before filling missing inputs

Training: 1950-2018. Validation: 2019-2021. Test: 2022-2024. Whole races stay together. The larger historical training period preserves the requested coverage but introduces era changes; validation/test measure performance on later seasons.

Input groups are grid, recent form, prior standings, constructor, circuit, and calendar/age. Year and round are known before the race. Constructor/circuit categories are encoded using training categories, with an explicit audit flag for unseen IDs. All constant training columns are omitted from predictors, including unseen-ID flags that are always zero in training. This prevents a later flag from activating untrained neural-network weights; unseen IDs use an all-zero known-category block. No identity is manually merged across rebrands. Historical form/standings provide information beyond identity; ablations measure each group's additional value.

In [ ]:
NUMERIC_FEATURES = ['grid_position', 'driver_prior_entries', 'driver_prior5_podium_rate',
    'constructor_prior_races', 'constructor_prior5_podium_rate',
    'driver_standing_prior_points', 'driver_standing_prior_position',
    'constructor_standing_prior_points', 'constructor_standing_prior_position',
    'driver_age_years', 'year', 'round']
BINARY_FEATURES = ['grid_is_zero']
CATEGORICAL_FEATURES = ['constructorId', 'circuitId']
FEATURE_COLUMNS = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN_INPUTS = {'positionOrder', 'podium', 'position', 'statusId', 'status', 'laps',
                    'points', 'pit_stops', 'entry_type', 'grid_zero_reason_pre_race',
                    'qualifying_position', 'q1_gap_pct'}
assert not set(FEATURE_COLUMNS) & FORBIDDEN_INPUTS
modeling['split'] = 'train'
modeling.loc[modeling['year'].between(2019, 2021), 'split'] = 'validation'
modeling.loc[modeling['year'].between(2022, 2024), 'split'] = 'test'
split = modeling['split']
X_train_raw = modeling.loc[split.eq('train'), FEATURE_COLUMNS].copy()
X_val_raw = modeling.loc[split.eq('validation'), FEATURE_COLUMNS].copy()
X_test_raw = modeling.loc[split.eq('test'), FEATURE_COLUMNS].copy()
y_train = modeling.loc[split.eq('train'), 'podium'].copy()
y_val = modeling.loc[split.eq('validation'), 'podium'].copy()
y_test = modeling.loc[split.eq('test'), 'podium'].copy()
split_summary = modeling.groupby('split').agg(rows=('raceId', 'size'), races=('raceId', 'nunique'),
                                             first_year=('year', 'min'), last_year=('year', 'max'))
display(split_summary)
race_splits = modeling.groupby('raceId')['split'].nunique()
assert race_splits.eq(1).all()
assert modeling.loc[split.eq('train'), 'date'].max() < modeling.loc[split.eq('validation'), 'date'].min()
assert modeling.loc[split.eq('validation'), 'date'].max() < modeling.loc[split.eq('test'), 'date'].min()

### 3.3 Training-only evidence for features

Grid summarizes starting advantage; prior form and standings summarize established performance. Identity can reflect stable team/venue patterns; age and calendar capture pre-race context. Training-only plots motivate these hypotheses. Experiments later remove each group rather than assume every proposed group helps. These associations are not causal.

In [ ]:
train_eda = modeling[split.eq('train')].copy()
train_eda['grid_band'] = pd.cut(train_eda['grid_position'], [0, 3, 6, 10, 50], labels=['1-3', '4-6', '7-10', '11+'])
train_eda['driver_form_band'] = pd.cut(train_eda['driver_prior5_podium_rate'], [-0.01, 0, 0.2, 0.4, 1.01],
                                       labels=['0%', '1-20%', '21-40%', '>40%'])
evidence_tables = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feature in zip(axes, ['grid_band', 'driver_form_band', 'grid_is_zero']):
    table = train_eda.groupby(feature, observed=True)['podium'].agg(['size', 'mean'])
    evidence_tables[feature] = table
    ax.bar(table.index.astype(str), 100 * table['mean'], color='#287D8E')
    ax.set_title(feature.replace('_', ' '))
    ax.set_ylabel('Training podium rate (%)')
    display(table.round(3))
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/3_training_feature_evidence.png', dpi=150)
plt.show()

### 3.4 Historical filling and scaling after the split

For each date, use only **observed numeric input values from earlier race dates** to calculate a mean and population standard deviation. Fill a missing value with that earlier mean, and scale using those earlier statistics. This is stricter than a single training-period median: even the oldest training row cannot receive a fill derived from later training races. Races on the same date are processed as a batch. After an earlier validation/test race completes, its outcomes may update later lagged form and standings features and the history of those numeric inputs. No current or future race outcome enters a prediction. This is sequential pre-race evaluation; model weights, schema and thresholds remain frozen.

Before any history exists for an input, use standardized value 0 with explicit missing/no-history flags. A fixed raw fill 0 is only an internal placeholder; observed values are also neutralized on that cold-start date. Historical standard deviations below 1e-8 use scale 1. A zero fill is a model placeholder, not a claim that a driver has zero ability or starts from P0. Categories use training vocabulary only. A training-input-only constant-column filter fixes the predictor schema; flags remain available for audit even when unlearnable. Part 5 compares operation order, historical fills, feature removal and training-row policies. The final common strategy is selected on validation while preserving full validation/test coverage.


In [ ]:
category_levels = {c: sorted(X_train_raw[c].dropna().astype(int).unique()) for c in CATEGORICAL_FEATURES}

def encode_categories(frame):
    pieces = []
    for col, levels in category_levels.items():
        values = pd.Categorical(frame[col], categories=levels)
        block = pd.get_dummies(values, prefix=col, dtype=float)
        block.index = frame.index
        block[col + '_unseen'] = (~frame[col].isin(levels)).astype(float)
        pieces.append(block)
    return pd.concat(pieces, axis=1)

def history_stats(observed):
    if not len(observed):
        return pd.Series(0., index=NUMERIC_FEATURES), pd.Series(1., index=NUMERIC_FEATURES), pd.Series(True, index=NUMERIC_FEATURES)
    means = observed.mean()
    raw_scales = observed.std(ddof=0)
    scales = raw_scales.where(raw_scales.ge(1e-8), 1.0).fillna(1.0)
    return means.fillna(0), scales, means.isna()

def historical_fill_values(past_raw, method, means):
    if method == 'mean':
        return means
    if method == 'median':
        return past_raw[NUMERIC_FEATURES].median().fillna(means)
    if method != 'recent_weighted':
        raise ValueError('Unknown historical fill method')
    # Average within each earlier race date, then weight the last five dates 1..5.
    # Same-date races share a batch; missing dates do not count as observed values.
    recent = past_raw.groupby('date')[NUMERIC_FEATURES].mean().tail(5)
    weights = pd.Series(np.arange(1, len(recent) + 1, dtype=float), index=recent.index)
    numerator = recent.mul(weights, axis=0).sum()
    denominator = recent.notna().mul(weights, axis=0).sum().replace(0, np.nan)
    return (numerator / denominator).fillna(means)

def prepare_from_history(frame, past_raw, past_filled=None, variant='reference', fill_method='mean'):
    number = frame[NUMERIC_FEATURES].astype(float)
    observed = past_raw[NUMERIC_FEATURES].astype(float)
    means, scales, no_history = history_stats(observed)
    fill_values = historical_fill_values(past_raw, fill_method, means)
    filled = number.fillna(fill_values)
    if variant == 'impute_then_scale':
        # Earlier rows have already been imputed using their own earlier histories.
        scale_mean, scale_std, _ = history_stats(past_filled[NUMERIC_FEATURES])
        scaled = (filled - scale_mean) / scale_std
    else:
        # Reference: fit on earlier observed values, scale, then insert scaled historical means (zero).
        scaled = (filled - means) / scales
    # Without prior observations, use neutral standardized zero, even for an observed value.
    # Keep the no-history flag; never leave year=1950 on an otherwise standardized scale.
    scaled.loc[:, no_history] = 0.0
    binary = frame[BINARY_FEATURES].astype(float).copy()
    missing = number.isna().astype(float).add_suffix('_missing')
    if variant == 'flags_after_fill':
        missing = filled.isna().astype(float).add_suffix('_missing')
    # grid_is_zero already records the grid-zero condition; keep a distinct missing flag for missing inputs at inference.
    empty_history = pd.DataFrame(np.tile(no_history.astype(float), (len(frame), 1)),
                                 columns=[c + '_no_history' for c in NUMERIC_FEATURES], index=frame.index)
    encoded = encode_categories(frame)
    indicators = pd.concat([binary, missing, empty_history, encoded], axis=1)
    return pd.concat([scaled, indicators], axis=1).astype(float), filled

def chronological_prepare(table, variant='reference', fill_method='mean'):
    parts, filled_parts, history_parts = [], [], []
    for date, batch in table.groupby('date', sort=True):
        past = table.loc[table['date'].lt(date), FEATURE_COLUMNS + ['date']]
        past_filled = pd.concat(filled_parts) if filled_parts else pd.DataFrame(columns=NUMERIC_FEATURES, dtype=float)
        prepared, filled = prepare_from_history(batch[FEATURE_COLUMNS], past, past_filled, variant, fill_method)
        parts.append(prepared)
        filled_parts.append(filled)
        history_parts.append({'date': date, 'rows': len(batch), 'earlier_rows': len(past),
                              'latest_source_date': table.loc[table['date'].lt(date), 'date'].max()})
    return pd.concat(parts).sort_index(), pd.concat(filled_parts).sort_index(), pd.DataFrame(history_parts)

X_all_full, filled_all, preparation_audit = chronological_prepare(modeling)
# A constant column has no learnable effect. In particular, unseen-ID flags are
# always zero in training and would otherwise activate untrained NN weights later.
constant_training_columns = X_all_full.loc[X_train_raw.index].columns[
    X_all_full.loc[X_train_raw.index].nunique().le(1)].tolist()
ACTIVE_COLUMNS = X_all_full.columns.drop(constant_training_columns)
X_all = X_all_full[ACTIVE_COLUMNS].copy()
X_train, X_val, X_test = [X_all.loc[idx] for idx in [X_train_raw.index, X_val_raw.index, X_test_raw.index]]
assert np.isfinite(X_all.to_numpy()).all()
assert X_train.columns.equals(X_val.columns) and X_train.columns.equals(X_test.columns)
assert not set(X_all.columns) & FORBIDDEN_INPUTS
assert preparation_audit['latest_source_date'].dropna().lt(preparation_audit.loc[
    preparation_audit['latest_source_date'].notna(), 'date']).all()
print('Prepared matrix shapes:', X_train.shape, X_val.shape, X_test.shape)
print('Constant training columns retained only for audit:', constant_training_columns)
display(preparation_audit.head(3))
drift_table = pd.DataFrame({c: [X_all_full.loc[X_val.index, c + '_unseen'].mean(), X_all_full.loc[X_test.index, c + '_unseen'].mean()]
                           for c in CATEGORICAL_FEATURES}, index=['validation', 'test']) * 100
print('Unseen category rows (%):')
display(drift_table.round(2))


In [ ]:
# A future-tampering check: later data must not alter an earlier prefix's preparation.
dates = sorted(modeling['date'].unique())
probe = modeling[modeling['date'].isin(dates[:4])].copy()
changed = probe.copy()
changed.loc[changed['date'].eq(dates[3]), NUMERIC_FEATURES] = 999999.
earlier = probe['date'].lt(dates[3])
for method in ['mean', 'median', 'recent_weighted']:
    probe_x, _, _ = chronological_prepare(probe, fill_method=method)
    changed_x, _, _ = chronological_prepare(changed, fill_method=method)
    assert np.allclose(probe_x.loc[earlier], changed_x.loc[earlier])
# The same-date batch cannot change its own fill statistics.
batch = modeling[modeling['date'].eq(dates[1])][FEATURE_COLUMNS].copy()
past = modeling[modeling['date'].lt(dates[1])][FEATURE_COLUMNS + ['date']]
means_before, _, _ = history_stats(past[NUMERIC_FEATURES])
batch.iloc[0, batch.columns.get_loc('driver_age_years')] = np.nan
_, probe_filled = prepare_from_history(batch, past)
assert np.isclose(probe_filled.iloc[0]['driver_age_years'], means_before['driver_age_years'])
for method in ['median', 'recent_weighted']:
    expected_fill = historical_fill_values(past, method, means_before)['driver_age_years']
    _, probe_filled = prepare_from_history(batch, past, fill_method=method)
    assert np.isclose(probe_filled.iloc[0]['driver_age_years'], expected_fill)
print('Passed: split separation, unique grain, forbidden inputs, historical source dates, finite matrices, future tampering, same-date filling.')

## 4. Predictive modeling

The baseline is logistic regression with C=1. Compare logistic regression, random forest and a shallow neural network on identical rows and features. Keep the alternative settings visible. Average precision (AP, labeled PR-AUC in tables) is primary because podium entries are the minority class; F1 is secondary. ROC-AUC, precision and recall provide context. AP uses scikit-learn's average precision rather than trapezoidal integration.

Hyperparameters and probability thresholds are selected on validation only. For each candidate, select the validation-F1 threshold from distinct validation probabilities. Freeze model choices before final test evaluation. The neural network uses early stopping on validation loss; validation is a tuning set, not an independent uncertainty estimate. All fits use fixed seed 42; library and hardware changes can still change results.

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve, confusion_matrix)
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import copy

tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
versions = {'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
            'tensorflow': tf.__version__, 'keras': keras.__version__}
print(versions)

def best_threshold(labels, probability):
    precision, recall, thresholds = precision_recall_curve(labels, probability)
    denominator = precision[:-1] + recall[:-1]
    values = np.divide(2 * precision[:-1] * recall[:-1], denominator,
                       out=np.zeros_like(denominator), where=denominator > 0)
    return float(thresholds[np.flatnonzero(values == values.max())[-1]])

def metrics(labels, probability, threshold):
    prediction = np.asarray(probability) >= threshold
    return {'PR-AUC': average_precision_score(labels, probability),
            'ROC-AUC': roc_auc_score(labels, probability),
            'F1': f1_score(labels, prediction, zero_division=0),
            'Precision': precision_score(labels, prediction, zero_division=0),
            'Recall': recall_score(labels, prediction, zero_division=0)}

def predict_probability(model, inputs):
    if isinstance(model, keras.Model):
        # Direct batched inference avoids starting a training/prediction data pipeline for every SHAP call.
        array = np.asarray(inputs, dtype=np.float32)
        return np.concatenate([np.asarray(model(array[i:i + 2048], training=False)).ravel()
                               for i in range(0, len(array), 2048)])
    return model.predict_proba(inputs)[:, 1]

def fit_model(family, settings, train_inputs, val_inputs):
    keras.utils.set_random_seed(42)
    train_labels = y_train.loc[train_inputs.index]
    val_labels = y_val.loc[val_inputs.index]
    assert train_inputs.columns.equals(val_inputs.columns)
    epochs = None
    history = None
    if family == 'Logistic regression':
        model = LogisticRegression(max_iter=2000, **settings)
        model.fit(train_inputs, train_labels)
    elif family == 'Random forest':
        model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=2, **settings)
        model.fit(train_inputs, train_labels)
    else:
        model = keras.Sequential([layers.Input(shape=(train_inputs.shape[1],))] +
            [layers.Dense(units, activation='relu') for units in settings['hidden']] +
            [layers.Dense(1, activation='sigmoid')])
        model.compile(optimizer='adam', loss='binary_crossentropy')
        stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        history = model.fit(np.asarray(train_inputs, dtype=np.float32), train_labels.to_numpy(),
            validation_data=(np.asarray(val_inputs, dtype=np.float32), val_labels.to_numpy()),
            epochs=150, batch_size=128, callbacks=[stop], verbose=0)
        epochs = len(history.history['loss'])
    train_probability = predict_probability(model, train_inputs)
    val_probability = predict_probability(model, val_inputs)
    threshold = best_threshold(val_labels, val_probability)
    return {'model': model, 'family': family, 'settings': copy.deepcopy(settings),
            'threshold': threshold, 'epochs': epochs, 'history': history,
            'best_epoch': (int(np.argmin(history.history['val_loss'])) + 1) if history is not None else None,
            'train_probability': train_probability, 'validation_probability': val_probability,
            'train_index': train_inputs.index.copy(), 'input_columns': train_inputs.columns.copy()}

fitted, attempt_rows = {}, []
def run_attempt(name, family, settings):
    result = fit_model(family, settings, X_train, X_val)
    fitted[name] = result
    for label, target in [('train', y_train), ('validation', y_val)]:
        probability = result[label + '_probability']
        attempt_rows.append({'model': name, 'family': family, 'split': label,
            'threshold': result['threshold'], 'epochs': result['epochs'],
            **metrics(target, probability, result['threshold'])})
    print(name, '| validation AP:', round(average_precision_score(y_val, result['validation_probability']), 4), flush=True)


### 4.1 Statistical baseline and logistic-regression variants

Logistic regression combines the inputs into a log-odds score. A smaller C increases regularization. It is a useful simple baseline but cannot freely learn interactions.

In [ ]:
run_attempt('LR baseline C=1', 'Logistic regression', {'C': 1.0})
for c in [0.1, 10.0]:
    run_attempt(f'LR C={c}', 'Logistic regression', {'C': c})
display(pd.DataFrame(attempt_rows).query("family == 'Logistic regression'").round(4))

### 4.2 Random-forest variants

A forest averages many decision trees and can capture nonlinear interactions. Restricting depth and requiring larger leaves can reduce fitting to historical peculiarities. All candidates use 100 trees and the same seed.

In [ ]:
run_attempt('RF default', 'Random forest', {})
for depth in [5, 10]:
    run_attempt(f'RF depth={depth}, leaf=5', 'Random forest', {'max_depth': depth, 'min_samples_leaf': 5})
display(pd.DataFrame(attempt_rows).query("family == 'Random forest'").round(4))

### 4.3 Shallow neural-network variants

Compare one hidden layer of 16 units with two layers of 32 and 16 units. ReLU learns nonlinear combinations; sigmoid produces a podium probability. Early stopping restores the weights with the lowest validation loss. Compare their actual ranking scores rather than assume extra depth helps.

In [ ]:
for hidden in [(16,), (32, 16)]:
    run_attempt(f'NN hidden={hidden}', 'Neural network', {'hidden': hidden})
display(pd.DataFrame(attempt_rows).query("family == 'Neural network'").round(4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, item in fitted.items():
    if item['history'] is not None:
        for ax, key in zip(axes, ['loss', 'val_loss']):
            ax.plot(item['history'].history[key], label=name)
            ax.set_title(key.replace('_', ' '))
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Binary cross-entropy')
            ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/4_nn_learning_curves.png', dpi=150)
plt.show()

In [ ]:
attempt_table = pd.DataFrame(attempt_rows)
validation_attempts = attempt_table[attempt_table['split'].eq('validation')].set_index('model')
selected = {family: group['PR-AUC'].idxmax() for family, group in validation_attempts.groupby('family', sort=False)}
best_name = validation_attempts.loc[list(selected.values()), 'PR-AUC'].idxmax()
selection_snapshot = (selected.copy(), best_name)
display(attempt_table.round(4))
print('Selected variants:', selected)
print('Preferred model selected before test:', best_name)

## 5. Experiments for all three model families

### 5.1 Remove one feature group at a time

For each selected family design, remove one complete group and train again. Keep rows, labels, settings, seed and stopping policy fixed. The reference is that family's full-feature fit. Each variant selects its F1 threshold on validation; AP does not depend on a threshold. Report all five validation metrics and training AP. These are validation tuning comparisons; the held-out test does not select a group or variant. No values are scrambled in Part 5.

The effect is the value of a group **given the remaining groups**, not its causal effect. Correlated groups can substitute for each other. A single training seed cannot establish statistical significance.

In [ ]:
FEATURE_GROUPS = {
    'Grid': [c for c in X_train if c.startswith('grid_')],
    'Recent form': [c for c in X_train if c.startswith(('driver_prior', 'constructor_prior'))],
    'Prior standings': [c for c in X_train if c.startswith(('driver_standing_', 'constructor_standing_'))],
    'Constructor identity': [c for c in X_train if c.startswith('constructorId_')],
    'Circuit identity': [c for c in X_train if c.startswith('circuitId_')],
    'Calendar and age': [c for c in X_train if c.startswith(('year', 'round', 'driver_age_years'))],
}
grouped_columns = [c for columns in FEATURE_GROUPS.values() for c in columns]
assert len(grouped_columns) == len(set(grouped_columns)) and set(grouped_columns) == set(X_train)

def experiment_row(family, experiment, result, removed=0):
    return {'family': family, 'experiment': experiment, 'removed columns': removed,
            'train rows': len(result['train_index']), 'validation rows': len(y_val),
            'input columns': len(result['input_columns']),
            'train PR-AUC': average_precision_score(y_train.loc[result['train_index']], result['train_probability']),
            'threshold': result['threshold'], 'epochs': result['epochs'],
            **{'validation ' + k: v for k, v in metrics(y_val, result['validation_probability'], result['threshold']).items()}}

ablation_rows, reusable_ablations = [], {}
for family, name in selected.items():
    reference = fitted[name]
    ablation_rows.append(experiment_row(family, 'None (all features)', reference))
    for group, removed in FEATURE_GROUPS.items():
        keep = X_train.columns.drop(removed)
        result = fit_model(family, reference['settings'], X_train[keep], X_val[keep])
        if group in ['Prior standings', 'Constructor identity']:
            reusable_ablations[(family, group)] = result
        ablation_rows.append(experiment_row(family, group, result, len(removed)))
        print('Finished', family, '/', group, flush=True)
ablation_table = pd.DataFrame(ablation_rows)
reference_ap = ablation_table[ablation_table['experiment'].eq('None (all features)')].set_index('family')['validation PR-AUC']
ablation_table['AP change vs reference'] = ablation_table['validation PR-AUC'] - ablation_table['family'].map(reference_ap)
display(ablation_table.round(4))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, family in zip(axes, selected):
    chart = ablation_table[ablation_table['family'].eq(family) & ablation_table['removed columns'].gt(0)].sort_values('AP change vs reference')
    ax.barh(chart['experiment'], chart['AP change vs reference'],
            color=['#B95D4C' if v < 0 else '#287D8E' for v in chart['AP change vs reference']])
    ax.axvline(0, color='#334155', lw=0.8)
    ax.set_title(family)
    ax.set_xlabel('Validation AP change after removal')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/5_ablation_all_models.png', dpi=150)
plt.show()

### 5.2 Two preprocessing-order experiments for each family

Reference: record missingness flags before filling; estimate numeric scaling from earlier **observed** values, then scale and insert the scaled historical mean (zero).

1. **Impute then fit scaling:** fill each historical row from its own earlier races, then estimate scaling from those earlier **filled** rows. This changes which historical values determine the scaler. Fills and scales still exclude the target date and later dates.
2. **Fill then create missing flags:** reverse missing-flag creation and filling. The numeric values are identical to reference, but formerly missing values no longer produce a missing flag. This tests the information lost when indicators are created too late. No-history flags and the known grid-zero flag are preserved.

A third comparison restores the legacy raw-scale cold start on the same reference inputs to measure its numerical effect; it is not counted as one of the two order experiments. No records are scrambled.

Both required order trials are operation-order comparisons on the same numeric input definitions. The earlier draft's encode-and-scale-all comparison changed scaling scope as well as order; these experiments replace it. Each comparison is repeated for logistic regression, random forest and the shallow NN, with the family's selected settings fixed.


In [ ]:
preprocessing_inputs = {}
for key in ['impute_then_scale', 'flags_after_fill']:
    prepared, _, _ = chronological_prepare(modeling, variant=key)
    preprocessing_inputs[key] = (prepared.loc[X_train.index, ACTIVE_COLUMNS], prepared.loc[X_val.index, ACTIVE_COLUMNS])
# Additional cold-start comparison: same records, inputs and model settings; no shuffling.
legacy_cold = X_all_full.copy()
for column in NUMERIC_FEATURES:
    cold = legacy_cold[column + '_no_history'].eq(1)
    legacy_cold.loc[cold, column] = filled_all.loc[cold, column]
preprocessing_inputs['legacy_cold_start'] = (legacy_cold.loc[X_train.index, ACTIVE_COLUMNS], legacy_cold.loc[X_val.index, ACTIVE_COLUMNS])
order_rows = []
for family, name in selected.items():
    reference = fitted[name]
    order_rows.append(experiment_row(family, 'Reference', reference))
    for key, label in [('impute_then_scale', 'Impute then fit scaling'),
                       ('flags_after_fill', 'Fill then create missing flags'),
                       ('legacy_cold_start', 'Legacy raw-scale cold start')]:
        train_inputs, val_inputs = preprocessing_inputs[key]
        result = fit_model(family, reference['settings'], train_inputs, val_inputs)
        order_rows.append(experiment_row(family, label, result))
        print('Finished', family, '/', label, flush=True)
order_table = pd.DataFrame(order_rows)
order_table['AP change vs reference'] = order_table['validation PR-AUC'] - order_table['family'].map(reference_ap)
display(order_table.round(4))
order_plot = order_table[order_table['experiment'].ne('Reference')].pivot(
    index='family', columns='experiment', values='AP change vs reference')
order_plot.plot.barh(figsize=(10, 4.5), color=['#287D8E', '#B95D4C', '#8798A5'])
plt.axvline(0, color='#334155', lw=0.8)
plt.xlabel('Validation AP change versus reference')
plt.ylabel('')
plt.title('Preprocessing-order comparisons: all three models')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/5_preprocessing_all_models.png', dpi=150)
plt.show()


In [ ]:
experiment_conclusions = []
for family in selected:
    effects = ablation_table[ablation_table['family'].eq(family) & ablation_table['removed columns'].gt(0)]
    strongest = effects.loc[effects['AP change vs reference'].idxmin()]
    best_order = order_table[order_table['family'].eq(family)].sort_values('validation PR-AUC', ascending=False).iloc[0]
    text = (f'{family}: largest observed AP loss after removing {strongest["experiment"]} '
            f'({strongest["AP change vs reference"]:+.4f}). Best preprocessing row: '
            f'{best_order["experiment"]} (validation AP {best_order["validation PR-AUC"]:.4f}).')
    experiment_conclusions.append(text)
display(Markdown('\n\n'.join(experiment_conclusions) +
    '\n\nThese are single-seed validation comparisons. They do not establish causal effects or significant superiority. '
    'These full-feature reference comparisons are preserved; the next stages choose a common final strategy on validation.'))

### 5.3 Compare historical fills on the same entries

Compare the earlier observed **mean**, earlier observed **median**, and a **recent weighted mean**. For the weighted option, calculate each input's mean within each of the last five strictly earlier race dates and weight those dates 1, 2, ..., 5 from oldest to newest. An entirely unobserved recent input falls back to its earlier overall mean; no earlier observations means the fixed cold-start zero. All methods retain missing/no-history flags. Scaling always uses the same earlier observed mean and standard deviation, isolating the fill choice.

The focused search has three stages: fill method, feature combination, then training-row policy. At each stage choose the largest **mean validation AP across the three families**, with equal family weights and the listed order breaking an exact tie. This selects one common pipeline rather than different inputs for each model. Family settings from Section 4 stay fixed. This is a small sequential search, not an exhaustive search of all interactions. Validation is reused for tuning, so its scores are optimistic selection evidence. Previously viewed test results are excluded from every choice; the later-era test is not a newly untouched dataset after project revisions.

In [ ]:
reference_selected = selected.copy()
reference_groups = copy.deepcopy(FEATURE_GROUPS)
reference_columns = X_train.columns.copy()
reference_fitted = fitted.copy()
fill_labels = {'mean': 'Historical mean', 'median': 'Historical median',
               'recent_weighted': 'Recent weighted mean (five dates)'}
fill_matrices = {'mean': X_all.copy()}
fill_filled = {'mean': filled_all.copy()}
fill_results, fill_rows, common_selection_parts = {}, [], []

def choose_common(table, stage):
    scores = table.groupby('experiment', sort=False).agg(
        mean_validation_AP=('validation PR-AUC', 'mean'),
        minimum_family_AP=('validation PR-AUC', 'min'), families=('family', 'nunique')).reset_index()
    assert scores['families'].eq(3).all()
    winner = scores.loc[scores['mean_validation_AP'].idxmax(), 'experiment']
    scores['stage'], scores['selected'] = stage, scores['experiment'].eq(winner)
    common_selection_parts.append(scores)
    display(scores.round(5))
    return winner

for method, label in fill_labels.items():
    if method != 'mean':
        prepared, filled_variant, _ = chronological_prepare(modeling, fill_method=method)
        fill_filled[method] = filled_variant
        fill_matrices[method] = prepared[reference_columns]
    matrix = fill_matrices[method]
    for family, name in reference_selected.items():
        result = reference_fitted[name] if method == 'mean' else fit_model(
            family, reference_fitted[name]['settings'], matrix.loc[X_train_raw.index], matrix.loc[X_val_raw.index])
        fill_results[(method, family)] = result
        fill_rows.append(experiment_row(family, label, result))
        print('Fill comparison:', family, '/', label, flush=True)
fill_table = pd.DataFrame(fill_rows)
fill_table['AP change vs historical mean'] = fill_table['validation PR-AUC'] - fill_table['family'].map(reference_ap)
display(fill_table.round(4))
selected_fill_label = choose_common(fill_table, 'Historical fill')
selected_fill = next(key for key, label in fill_labels.items() if label == selected_fill_label)
fill_table.pivot(index='family', columns='experiment', values='validation PR-AUC').plot.barh(
    figsize=(11, 4.5), color=['#287D8E', '#B95D4C', '#8798A5'])
plt.xlabel('Validation AP (same entries, scaling and model settings)')
plt.ylabel('')
plt.title('Historical filling comparison: all three families')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/5_historical_fills.png', dpi=150)
plt.show()


### 5.4 Test the promising feature removals together

The initial one-group comparisons motivate four candidates: retain all groups, remove prior standings, remove constructor identity, or remove both. Test each on the chosen fill method, with identical training/validation entries and fixed family settings. Removing both needs a separate fit: individual improvements cannot simply be added. Existing fits are reused only when the exact inputs and settings agree. Keep the other groups, including the grid-zero code. Select the common feature set by mean validation AP across families.

In [ ]:
feature_options = {'All feature groups': [], 'Remove prior standings': ['Prior standings'],
    'Remove constructor identity': ['Constructor identity'],
    'Remove standings and constructor': ['Prior standings', 'Constructor identity']}
feature_results, feature_columns, feature_rows = {}, {}, []
chosen_fill_matrix = fill_matrices[selected_fill]
for label, dropped_groups in feature_options.items():
    removed = [column for group in dropped_groups for column in reference_groups[group]]
    keep = reference_columns.drop(removed)
    feature_columns[label] = keep
    for family, name in reference_selected.items():
        if not dropped_groups:
            result = fill_results[(selected_fill, family)]
        elif selected_fill == 'mean' and len(dropped_groups) == 1:
            result = reusable_ablations[(family, dropped_groups[0])]
        else:
            result = fit_model(family, reference_fitted[name]['settings'],
                chosen_fill_matrix.loc[X_train_raw.index, keep], chosen_fill_matrix.loc[X_val_raw.index, keep])
        feature_results[(label, family)] = result
        feature_rows.append(experiment_row(family, label, result, len(removed)))
        print('Combined features:', family, '/', label, flush=True)
feature_combination_table = pd.DataFrame(feature_rows)
feature_reference = feature_combination_table[feature_combination_table['experiment'].eq('All feature groups')].set_index('family')['validation PR-AUC']
feature_combination_table['AP change vs all groups'] = feature_combination_table['validation PR-AUC'] - feature_combination_table['family'].map(feature_reference)
display(feature_combination_table.round(4))
selected_features = choose_common(feature_combination_table, 'Feature combination')
candidate_columns = feature_columns[selected_features]
feature_combination_table.pivot(index='family', columns='experiment', values='validation PR-AUC').plot.barh(
    figsize=(12, 5), color=['#8798A5', '#287D8E', '#B95D4C', '#A38B54'])
plt.xlabel('Validation AP (same entries and chosen historical fill)')
plt.ylabel('')
plt.title('Single and combined feature removals')
plt.tight_layout()
plt.savefig(f'{FIG_PATH}/5_combined_feature_removals.png', dpi=150)
plt.show()

### 5.5 Test training-row exclusion; preserve prediction coverage

Compare keeping all training entries, dropping only entries with **unrecorded retained inputs**, and a stricter complete-input sensitivity. Determine missingness on the original pre-race inputs, before filling. Missing grid rank with grid_is_zero=1 is a known code and never triggers exclusion. Missing prior-form rates with zero earlier entries/races, and absent standings for an entity's first recorded race date of a season, are structural no-history cases. The first exclusion policy keeps them. The strict sensitivity also removes these no-history cases; its loss of debutants and season-opening entries must be weighed against its validation result.

Only retained numeric inputs and retained category IDs determine exclusion. Columns removed in the previous stage cannot discard a row. No current-race target, status or laps determines the mask. **All validation and test entries are kept and filled in every policy**, ensuring comparable scores and full prediction coverage. Excluded training examples may still supply observed historical pre-race inputs at subsequent dates, just as recorded earlier races remain available at deployment. After exclusion, constant training columns are removed using only the retained training inputs; all three families receive the same resulting schema.

In [ ]:
retained_numeric = [c for c in NUMERIC_FEATURES if c in candidate_columns]
retained_categories = [c for c in CATEGORICAL_FEATURES if any(p.startswith(c + '_') for p in candidate_columns)]
raw_missing = modeling[retained_numeric + retained_categories].isna()
known_code = pd.DataFrame(False, index=modeling.index, columns=raw_missing.columns)
structural = known_code.copy()
if 'grid_position' in raw_missing:
    known_code['grid_position'] = modeling['grid_is_zero'].eq(1) & raw_missing['grid_position']
for rate, count in [('driver_prior5_podium_rate', 'driver_prior_entries'),
                    ('constructor_prior5_podium_rate', 'constructor_prior_races')]:
    if rate in raw_missing:
        structural[rate] = modeling[count].eq(0) & raw_missing[rate]
for entity, prefix in [('driverId', 'driver'), ('constructorId', 'constructor')]:
    first_date = modeling.groupby(['year', entity])['date'].transform('min')
    for suffix in ['points', 'position']:
        column = prefix + '_standing_prior_' + suffix
        if column in raw_missing:
            structural[column] = modeling['date'].eq(first_date) & raw_missing[column]
unrecorded = raw_missing & ~known_code & ~structural
strict_missing = raw_missing & ~known_code
missing_audit_rows = []
for split_label, indices in [('train', X_train_raw.index), ('validation', X_val_raw.index), ('test', X_test_raw.index)]:
    for column in raw_missing:
        missing_audit_rows.append({'split': split_label, 'feature': column, 'entries': len(indices),
            'missing': int(raw_missing.loc[indices, column].sum()),
            'known grid code': int(known_code.loc[indices, column].sum()),
            'structural no history': int(structural.loc[indices, column].sum()),
            'unrecorded': int(unrecorded.loc[indices, column].sum())})
missing_input_audit = pd.DataFrame(missing_audit_rows)
display(missing_input_audit)
training_policies = {
    'Keep all training entries': pd.Series(True, index=X_train_raw.index),
    'Drop unrecorded training inputs': ~unrecorded.loc[X_train_raw.index].any(axis=1),
    'Strict complete training inputs': ~strict_missing.loc[X_train_raw.index].any(axis=1)}
row_results, row_columns, row_rows, row_population_rows = {}, {}, [], []
for policy, keep_mask in training_policies.items():
    kept_index = keep_mask.index[keep_mask]
    varied = chosen_fill_matrix.loc[kept_index, candidate_columns].nunique().gt(1)
    columns = candidate_columns[varied]
    row_columns[policy] = columns
    for split_label, source_index in [('train', X_train_raw.index), ('validation', X_val_raw.index), ('test', X_test_raw.index)]:
        retained_index = kept_index if split_label == 'train' else source_index
        removed_index = source_index.difference(retained_index)
        row_population_rows.append({'policy': policy, 'split': split_label, 'original rows': len(source_index),
            'retained rows': len(retained_index), 'removed rows': len(removed_index),
            'retained %': 100 * len(retained_index) / len(source_index),
            'retained podium rate': modeling.loc[retained_index, 'podium'].mean() if split_label == 'train' else np.nan,
            'removed podium rate': modeling.loc[removed_index, 'podium'].mean() if split_label == 'train' else np.nan,
            'input columns': len(columns)})
    for family, name in reference_selected.items():
        if kept_index.equals(X_train_raw.index) and columns.equals(candidate_columns):
            result = feature_results[(selected_features, family)]
        else:
            result = fit_model(family, reference_fitted[name]['settings'],
                chosen_fill_matrix.loc[kept_index, columns], chosen_fill_matrix.loc[X_val_raw.index, columns])
        row_results[(policy, family)] = result
        row_rows.append(experiment_row(family, policy, result, len(reference_columns) - len(columns)))
        print('Training-row policy:', family, '/', policy, '/', len(kept_index), 'rows', flush=True)
row_policy_table = pd.DataFrame(row_rows)
row_population_table = pd.DataFrame(row_population_rows)
row_reference = row_policy_table[row_policy_table['experiment'].eq('Keep all training entries')].set_index('family')['validation PR-AUC']
row_policy_table['AP change vs keep all'] = row_policy_table['validation PR-AUC'] - row_policy_table['family'].map(row_reference)
display(row_population_table.round(4))
display(row_policy_table.round(4))
selected_row_policy = choose_common(row_policy_table, 'Training-row policy')
common_selection_table = pd.concat(common_selection_parts, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
row_policy_table.pivot(index='family', columns='experiment', values='validation PR-AUC').plot.barh(
    ax=axes[0], color=['#287D8E', '#8798A5', '#B95D4C'])
axes[0].set(xlabel='Validation AP (all validation entries retained)', ylabel='')
coverage = row_population_table[row_population_table['split'].eq('train')]
axes[1].barh(coverage['policy'], coverage['retained %'], color='#287D8E')
axes[1].set(xlabel='Training entries retained (%)', ylabel='', xlim=(0, 105))
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/5_training_row_policies.png', dpi=150)
plt.show()

### 5.6 Freeze the common final pipeline

Use the selected fill, feature set and training-row policy for all three families. Select the preferred family by its validation AP on these common inputs. Family hyperparameters remain those selected in Section 4; this focused experiment does not claim to re-optimize every setting after feature changes. Fit the C=1 logistic baseline on the same final inputs as well. Cache the pipeline, schemas, thresholds and model choices before test prediction; Part 6 and inference must use this same final pipeline.

In [ ]:
ACTIVE_COLUMNS = row_columns[selected_row_policy]
final_train_index = training_policies[selected_row_policy].index[training_policies[selected_row_policy]]
X_all = chosen_fill_matrix[ACTIVE_COLUMNS].copy()
X_train, X_val, X_test = [X_all.loc[index] for index in [final_train_index, X_val_raw.index, X_test_raw.index]]
final_fitted, final_selected = {}, {}
for family, name in reference_selected.items():
    final_name = 'Final ' + name
    final_fitted[final_name] = row_results[(selected_row_policy, family)]
    final_selected[family] = final_name
baseline_name = 'LR baseline C=1 (final inputs)'
final_fitted[baseline_name] = fit_model('Logistic regression', {'C': 1.0}, X_train, X_val)
fitted, selected = final_fitted, final_selected
y_train = modeling.loc[final_train_index, 'podium'].copy()
best_name = max(selected.values(), key=lambda name: average_precision_score(y_val, fitted[name]['validation_probability']))
selection_snapshot = (selected.copy(), best_name)
FEATURE_GROUPS = {group: [c for c in columns if c in ACTIVE_COLUMNS] for group, columns in reference_groups.items()}
FEATURE_GROUPS = {group: columns for group, columns in FEATURE_GROUPS.items() if columns}
final_pipeline = {'fill_method': selected_fill, 'fill_description': selected_fill_label,
    'feature_option': selected_features, 'dropped_groups': feature_options[selected_features],
    'row_policy': selected_row_policy, 'training_rows': len(X_train), 'validation_rows': len(X_val),
    'test_rows': len(X_test), 'features': len(ACTIVE_COLUMNS), 'feature_groups': list(FEATURE_GROUPS),
    'feature_columns': list(ACTIVE_COLUMNS), 'selection_rule': 'Equal-weight mean validation AP across three families, stagewise'}
final_pipeline_snapshot = json.dumps(final_pipeline, sort_keys=True)
assert len(X_val) == len(X_val_raw) and len(X_test) == len(X_test_raw)
assert X_train.columns.equals(X_val.columns) and X_train.columns.equals(X_test.columns)
pipeline_conclusion = (f'Common final pipeline: {selected_fill_label}; {selected_features}; {selected_row_policy}. '
    f'Training retains {len(X_train):,}/{len(X_train_raw):,} entries; all {len(X_val):,} validation '
    f'and {len(X_test):,} test entries remain. {len(ACTIVE_COLUMNS)} prepared input columns. '
    'The three stages maximize equal-weight mean validation AP; all families use the same final inputs. '
    'These single-seed tuning gains have no significance claim, and untested interactions may behave differently.')
experiment_conclusions.append(pipeline_conclusion)
display(Markdown(pipeline_conclusion))
print('Frozen preferred model:', best_name)

### Final preprocessing evidence and final NN learning history

Show actual retained-input values before filling, after the chosen historical fill,
and after chronological scaling. Training-only plots and a row-level example
distinguish this from the earlier missing-code cleaning plots. The cold-start
audit must show standardized year zero, with its no-history flag set. Final NN
curves use the selected input schema, and distinguish epochs run from restored
best epoch. These diagnostics do not select from test scores.


In [ ]:
chosen_filled = fill_filled[selected_fill]
retained_numbers = [c for c in NUMERIC_FEATURES if c in ACTIVE_COLUMNS]
preprocessing_value_rows = []
for column in retained_numbers:
    before = modeling.loc[X_train.index, column]
    filled = chosen_filled.loc[X_train.index, column]
    scaled = X_train[column]
    preprocessing_value_rows.append({'feature': column, 'missing before': int(before.isna().sum()),
        'raw min': before.min(), 'raw max': before.max(), 'filled min': filled.min(),
        'filled max': filled.max(), 'scaled min': scaled.min(), 'scaled max': scaled.max()})
preprocessing_values = pd.DataFrame(preprocessing_value_rows)
display(preprocessing_values.round(4))
first_date = modeling['date'].min()
cold_indices = modeling.index[modeling['date'].eq(first_date)]
cold_start_audit = modeling.loc[cold_indices, ['raceId', 'driverId', 'date', 'year']].copy()
cold_start_audit['prepared year'] = X_all.loc[cold_indices, 'year']
cold_start_audit['year no history'] = chosen_fill_matrix.loc[cold_indices, 'year_no_history']
assert cold_start_audit['prepared year'].eq(0).all()
display(cold_start_audit.head())
candidates = modeling.index[(modeling['year'].between(2000, 2018)) & modeling['grid_position'].isna()]
example_index = candidates[0]
example_date = modeling.loc[example_index, 'date']
example_past = modeling.loc[modeling['date'].lt(example_date)]
example_mean, example_scale, example_empty = history_stats(example_past[NUMERIC_FEATURES])
example_fill = historical_fill_values(example_past, selected_fill, example_mean)
preprocessing_example = pd.DataFrame({'feature': retained_numbers,
    'raw value': modeling.loc[example_index, retained_numbers].to_numpy(),
    'historical fill': example_fill[retained_numbers].to_numpy(),
    'earlier mean': example_mean[retained_numbers].to_numpy(),
    'earlier std': example_scale[retained_numbers].to_numpy(),
    'prepared value': X_all.loc[example_index, retained_numbers].to_numpy()})
display(preprocessing_example)
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for j, column in enumerate(['grid_position', 'driver_prior5_podium_rate', 'year']):
    axes[0, j].hist(modeling.loc[X_train.index, column].dropna(), bins=30, alpha=.6, label='Observed raw')
    axes[0, j].hist(chosen_filled.loc[X_train.index, column], bins=30, alpha=.4, label='After historical fill')
    axes[0, j].set(title=column, ylabel='Training entries')
    axes[0, j].legend(fontsize=8)
    axes[1, j].hist(X_train[column], bins=30, color='#287D8E')
    axes[1, j].set(xlabel='Final prepared value', ylabel='Training entries')
fig.suptitle('Final preprocessing: observed, filled and standardized training inputs')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/3_final_preprocessing_values.png', dpi=150)
plt.show()
final_nn_name = selected['Neural network']
final_nn = fitted[final_nn_name]
final_learning_summary = pd.DataFrame([{'model': final_nn_name, 'input columns': len(ACTIVE_COLUMNS),
    'epochs run': final_nn['epochs'], 'restored best epoch': final_nn['best_epoch']}])
display(final_learning_summary)
fig, ax = plt.subplots(figsize=(9, 4))
for key, label in [('loss', 'Train loss'), ('val_loss', 'Validation loss')]:
    ax.plot(np.arange(1, final_nn['epochs'] + 1), final_nn['history'].history[key], label=label)
ax.axvline(final_nn['best_epoch'], color='#B95D4C', linestyle='--', label='Restored best epoch')
ax.set(xlabel='Epoch', ylabel='Binary cross-entropy', title='Final selected NN on final common inputs')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/4_final_nn_learning_curve.png', dpi=150)
plt.show()


### 5.7 Final held-out evaluation after experiments are fixed

The main models now use the common strategy selected in Sections 5.3-5.6. All choices above are frozen before these test predictions. Earlier project versions have already viewed this test period; no test result was used to choose the new strategy, but this is a revised later-era evaluation, not a fresh untouched holdout. No confidence interval is claimed.

In [ ]:
assert selection_snapshot == (selected, best_name)
assert final_pipeline_snapshot == json.dumps(final_pipeline, sort_keys=True)
final_rows = []
final_names = list(dict.fromkeys([baseline_name] + list(selected.values())))
for name in final_names:
    item = fitted[name]
    item['test_probability'] = predict_probability(item['model'], X_test)
    for label, target in [('train', y_train), ('validation', y_val), ('test', y_test)]:
        final_rows.append({'model': name, 'family': item['family'], 'split': label,
            'threshold': item['threshold'], 'rows': len(target), 'input columns': X_train.shape[1],
            **metrics(target, item[label + '_probability'], item['threshold'])})
final_table = pd.DataFrame(final_rows)
display(final_table.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, measure in zip(axes, ['PR-AUC', 'F1']):
    final_table[final_table['model'].isin(selected.values())].pivot(index='family', columns='split', values=measure)[
        ['train', 'validation', 'test']].plot.bar(ax=ax, color=['#A9B6C2', '#287D8E', '#B95D4C'])
    ax.set_ylabel(measure)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/4_model_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for family, name in selected.items():
    probability = fitted[name]['test_probability']
    precision, recall, _ = precision_recall_curve(y_test, probability)
    axes[0].plot(recall, precision, label=family)
    fpr, tpr, _ = roc_curve(y_test, probability)
    axes[1].plot(fpr, tpr, label=family)
axes[0].axhline(y_test.mean(), ls='--', color='grey', label='Test podium prevalence')
axes[0].set(xlabel='Recall', ylabel='Precision', title='Held-out test precision-recall curves')
axes[1].plot([0, 1], [0, 1], ls='--', color='grey')
axes[1].set(xlabel='False-positive rate', ylabel='True-positive rate', title='Held-out test ROC curves')
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/4_test_curves.png', dpi=150)
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, (family, name) in zip(axes, selected.items()):
    item = fitted[name]
    matrix = confusion_matrix(y_test, item['test_probability'] >= item['threshold'])
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set(title=family, xlabel='Predicted podium', ylabel='Actual podium')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/4_confusion_all_models.png', dpi=150)
plt.show()

## 6. Model explainability for all three families

### 6.1 Global group permutation importance

The TA restriction on scrambling applies to Part 5, not this required explanation method. Here each already fitted model remains frozen. Shuffle one group's columns together across validation rows and measure its AP loss, repeating five times. Moving whole one-hot blocks preserves a valid category pattern. Other cross-feature relationships may become unrealistic, and correlated inputs can substitute for each other. The result describes model reliance, not a causal effect and not the accuracy of a retrained reduced model.

In [ ]:
permutation_rows = []
for family, name in selected.items():
    item = fitted[name]
    baseline_ap = average_precision_score(y_val, item['validation_probability'])
    rng = np.random.default_rng(42)
    for group, columns in FEATURE_GROUPS.items():
        repeat_scores = []
        for repeat in range(5):
            shuffled = X_val.copy()
            row_order = rng.permutation(len(shuffled))
            shuffled.loc[:, columns] = X_val[columns].to_numpy()[row_order]
            repeat_scores.append(average_precision_score(y_val, predict_probability(item['model'], shuffled)))
        permutation_rows.append({'family': family, 'feature group': group,
            'baseline AP': baseline_ap, 'permuted AP mean': np.mean(repeat_scores),
            'AP drop mean': baseline_ap - np.mean(repeat_scores),
            'AP drop SD': np.std(repeat_scores, ddof=1)})
permutation_table = pd.DataFrame(permutation_rows)
display(permutation_table.round(4))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, family in zip(axes, selected):
    chart = permutation_table[permutation_table['family'].eq(family)].sort_values('AP drop mean')
    ax.barh(chart['feature group'], chart['AP drop mean'], xerr=chart['AP drop SD'], color='#287D8E', capsize=3)
    ax.axvline(0, color='#334155', lw=0.8)
    ax.set(title=family, xlabel='Validation AP drop (five shuffles)')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/6_permutation_all_models.png', dpi=150)
plt.show()

### 6.2 Global and local SHAP for every family

Use the same fixed sample of 20 validation entries and 12 training-background
entries for all models. Select the local entry by sorted (raceId, driverId)
without examining its outcome or prediction. Permutation SHAP attributes
differences from the background prediction across prepared columns. The
independent masker may create unrealistic cross-feature/category combinations.

The maximum-evaluation budget is `2 * (2 * prepared_columns + 1)` per entry.
SHAP omits invariant inputs; its effective forward/reverse cycle count therefore
depends on the varying columns for that entry. Export the actual budget,
varying-input count and effective cycles for every global/local row and family.
This is an approximate small-sample explanation, not a fixed two-cycle claim.
Prepared values are not original units. Neither global nor local explanations
establish causal effects or change model selection.


In [ ]:
import shap
versions['shap'] = shap.__version__
shap_background = X_train.sample(n=min(12, len(X_train)), random_state=42)
shap_sample = X_val.sample(n=min(20, len(X_val)), random_state=42)
local_index = modeling.loc[X_val.index, ['raceId', 'driverId']].sort_values(['raceId', 'driverId']).index[0]
local_inputs = X_val.loc[[local_index]]
shap_importance_rows, local_rows, xai_conclusions = [], [], []
shap_results = {}
shap_budget_rows = []

for model_number, (family, name) in enumerate(selected.items(), start=1):
    item = fitted[name]
    def explain_probability(values, frozen_model=item['model']):
        frame = pd.DataFrame(np.asarray(values, dtype=float), columns=X_train.columns)
        return predict_probability(frozen_model, frame)
    masker = shap.maskers.Independent(shap_background, max_samples=len(shap_background))
    explainer = shap.Explainer(explain_probability, masker, algorithm='permutation',
        feature_names=X_train.columns.tolist(), seed=42)
    evaluations = 2 * (2 * X_train.shape[1] + 1)
    for scope, frame in [('global', shap_sample), ('local', local_inputs)]:
        for index, row in frame.iterrows():
            varying = int((~masker.invariants(row.to_numpy())).any(axis=0).sum())
            shap_budget_rows.append({'family': family, 'scope': scope, 'row_index': int(index),
                'max_evals': evaluations, 'varying_inputs': varying,
                'effective_cycles': evaluations // (2 * varying + 1) if varying else 0})
    values = explainer(shap_sample, max_evals=evaluations, batch_size=1024)
    local = explainer(local_inputs, max_evals=evaluations, batch_size=1024)
    rebuilt = np.asarray(values.base_values).reshape(-1) + np.asarray(values.values).sum(axis=1)
    assert np.allclose(rebuilt, explain_probability(shap_sample), atol=1e-3)
    assert np.isclose(float(np.asarray(local.base_values).ravel()[0]) + np.asarray(local.values).sum(),
                      float(explain_probability(local_inputs)[0]), atol=1e-3)
    shap.plots.beeswarm(values, max_display=12, show=False)
    plt.title('Global SHAP: ' + family)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/6_shap_global_{model_number}.png', dpi=150, bbox_inches='tight')
    plt.show()
    shap.plots.waterfall(local[0], max_display=12, show=False)
    plt.title('Local SHAP: ' + family)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/6_shap_local_{model_number}.png', dpi=150, bbox_inches='tight')
    plt.show()
    importance = pd.Series(np.abs(values.values).mean(axis=0), index=X_train.columns).sort_values(ascending=False)
    for feature, value in importance.items():
        shap_importance_rows.append({'family': family, 'feature': feature, 'mean absolute SHAP': float(value)})
    effects = pd.Series(local.values.ravel(), index=X_train.columns)
    local_rows.append({'family': family, 'raceId': int(modeling.loc[local_index, 'raceId']),
        'driverId': int(modeling.loc[local_index, 'driverId']),
        'background probability': float(np.asarray(local.base_values).ravel()[0]),
        'predicted probability': float(explain_probability(local_inputs)[0]),
        'largest upward feature': effects.idxmax(), 'largest upward effect': effects.max(),
        'largest downward feature': effects.idxmin(), 'largest downward effect': effects.min()})
    reliance = permutation_table[permutation_table['family'].eq(family)].sort_values('AP drop mean', ascending=False).iloc[0]
    xai_conclusions.append(f'{family}: largest mean permutation AP drop is {reliance["feature group"]} '
        f'({reliance["AP drop mean"]:.4f}); largest mean absolute encoded-column SHAP is {importance.index[0]}. '
        f'The fixed local prediction is {local_rows[-1]["predicted probability"]:.3f}, '
        f'with {effects.idxmax()} pushing upward and {effects.idxmin()} downward relative to its background.')
    shap_results[family] = {'global': values, 'local': local}
    print('Finished SHAP:', family, flush=True)
shap_importance_table = pd.DataFrame(shap_importance_rows)
local_table = pd.DataFrame(local_rows)
shap_budget_audit = pd.DataFrame(shap_budget_rows)
display(shap_budget_audit.groupby(['family', 'scope'])[['max_evals', 'varying_inputs', 'effective_cycles']].agg(['min', 'max']))
display(local_table.round(4))
display(Markdown('\n\n'.join(xai_conclusions) + '\n\nThese explanations are not causal effects.'))


## 7. Validated inference

`predict_pre_race(record, race_date, mode='post_selection')` accepts engineered
pre-race inputs. It validates scalar/finite values and domains before conversion,
reconciles grid representations, derives/checks calendar year, prepares with strictly
earlier dates, and rejects a nonfinite matrix or probability. Unknown categories
follow the trained all-zero known-category policy. Genuine missing inputs are allowed.

If both driver history count and last-five rate are supplied, the rate must be
a possible mean of `min(count, 5)` binary outcomes. Constructor form averages
race-level fractions and is not subject to this binary-mean constraint.
Positive grids are supported up to the largest positive training grid; zero
remains a valid code. This is the API's validated historical domain, not a
universal Formula 1 rule. A future larger grid requires a reviewed contract.

Default predictions must follow the **selection-data cutoff**, not just training.
Explicit `mode='retrospective'` permits validation demonstrations after training,
while labeling their later selection exposure. The examples use a complete test-period
entry and a partially missing copy chosen by key order, after selection is frozen.
Historical examples are not claims that this revised model was deployed at that date.


In [ ]:
from collections.abc import Mapping
training_end = modeling.loc[split.eq('train'), 'date'].max()
selection_end = modeling.loc[split.eq('validation'), 'date'].max()
SUPPORTED_GRID_MAX = int(X_train_raw['grid_position'].max())

def predict_pre_race(record, race_date, model_name=None, mode='post_selection'):
    if not isinstance(record, Mapping):
        raise ValueError('record must be a mapping of pre-race inputs.')
    if mode not in {'post_selection', 'retrospective'}:
        raise ValueError('mode must be post_selection or retrospective.')
    model_name = best_name if model_name is None else model_name
    if model_name not in fitted:
        raise ValueError('Unknown fitted model name')
    try:
        date = pd.Timestamp(race_date)
    except (TypeError, ValueError) as error:
        raise ValueError('Supply a valid race calendar date.') from error
    if pd.isna(date) or date.tzinfo is not None:
        raise ValueError('Supply a valid calendar date without a timezone.')
    date = date.normalize()  # Never let an intraday timestamp include its own race.
    if date <= training_end:
        raise ValueError('Prediction date must follow the fitted training period.')
    if mode == 'post_selection' and date <= selection_end:
        raise ValueError('Prediction date must follow the selection period; explicitly use retrospective mode for a demonstration.')
    unknown = set(record) - set(FEATURE_COLUMNS) - {'grid_raw', 'grid_zero_reason_pre_race'}
    if unknown:
        raise ValueError('Unknown or forbidden fields: ' + ', '.join(sorted(map(str, unknown))))

    def number(value, field, integer=False, lower=None, upper=None):
        if not pd.api.types.is_scalar(value):
            raise ValueError(field + ' must be a scalar.')
        if value is None or pd.isna(value):
            return np.nan
        if isinstance(value, (bool, np.bool_)) and field != 'grid_is_zero':
            raise ValueError(field + ' must be a number, not a boolean.')
        try:
            value = float(value)
        except (TypeError, ValueError) as error:
            raise ValueError(field + ' must be numeric or missing.') from error
        if not np.isfinite(value):
            raise ValueError(field + ' must be finite or missing.')
        if integer and not value.is_integer():
            raise ValueError(field + ' must be an integer.')
        if lower is not None and value < lower or upper is not None and value > upper:
            raise ValueError(field + ' is outside its valid domain.')
        return value

    values = {column: number(record.get(column, np.nan), column) for column in FEATURE_COLUMNS}
    ranks = ['grid_position', 'driver_standing_prior_position', 'constructor_standing_prior_position']
    counts = ['driver_prior_entries', 'constructor_prior_races']
    for column in ranks:
        values[column] = number(values[column], column, integer=True,
            lower=0 if column == 'grid_position' else 1,
            upper=SUPPORTED_GRID_MAX if column == 'grid_position' else None)
    for column in counts:
        values[column] = number(values[column], column, integer=True, lower=0)
    for column in ['driver_prior5_podium_rate', 'constructor_prior5_podium_rate']:
        values[column] = number(values[column], column, lower=0, upper=1)
    for column in CATEGORICAL_FEATURES:
        values[column] = number(values[column], column, integer=True, lower=1)
    values['round'] = number(values['round'], 'round', integer=True, lower=1)
    values['driver_age_years'] = number(values['driver_age_years'], 'driver_age_years', lower=0, upper=100)
    values['year'] = number(values['year'], 'year', integer=True, lower=1)
    if pd.notna(values['year']) and int(values['year']) != date.year:
        raise ValueError('year must agree with race_date.')
    values['year'] = float(date.year)
    flag = number(values['grid_is_zero'], 'grid_is_zero', integer=True, lower=0, upper=1)
    raw_grid = number(record.get('grid_raw', np.nan), 'grid_raw', integer=True, lower=0, upper=SUPPORTED_GRID_MAX)
    ordinal = values['grid_position']
    if pd.notna(raw_grid) and pd.notna(ordinal) and raw_grid != ordinal:
        raise ValueError('grid_raw and grid_position disagree.')
    supplied_grid = raw_grid if pd.notna(raw_grid) else ordinal
    if pd.notna(supplied_grid):
        expected_zero = float(supplied_grid == 0)
        if pd.notna(flag) and flag != expected_zero:
            raise ValueError('grid_is_zero contradicts the supplied grid.')
        values['grid_is_zero'] = expected_zero
        values['grid_position'] = supplied_grid if supplied_grid > 0 else np.nan
    else:
        values['grid_is_zero'] = 0.0 if pd.isna(flag) else flag
        values['grid_position'] = np.nan
    for rate, count in [('driver_prior5_podium_rate', 'driver_prior_entries'),
                        ('constructor_prior5_podium_rate', 'constructor_prior_races')]:
        if pd.notna(values[count]) and values[count] == 0 and pd.notna(values[rate]):
            raise ValueError(rate + ' must be missing when no prior history exists.')
    count, rate = values['driver_prior_entries'], values['driver_prior5_podium_rate']
    if pd.notna(count) and count > 0 and pd.notna(rate):
        window = min(int(count), 5)
        implied_podiums = rate * window
        if not np.isclose(implied_podiums, np.rint(implied_podiums), rtol=0, atol=1e-6):
            raise ValueError('Driver form must be a possible mean of min(prior entries, 5) binary podium outcomes.')
    frame = pd.DataFrame([values], columns=FEATURE_COLUMNS)
    past = modeling.loc[modeling['date'].lt(date), FEATURE_COLUMNS + ['date']]
    prepared, _ = prepare_from_history(frame, past, fill_method=selected_fill)
    prepared = prepared.loc[:, ACTIVE_COLUMNS]
    if not np.isfinite(prepared.to_numpy()).all():
        raise ValueError('Prepared inputs must be finite.')
    item = fitted[model_name]
    probability = float(predict_probability(item['model'], prepared)[0])
    if not np.isfinite(probability) or not 0 <= probability <= 1:
        raise RuntimeError('Fitted model returned an invalid probability.')
    return {'model': model_name, 'mode': mode, 'podium probability': probability,
        'threshold': item['threshold'], 'predicted podium': bool(probability >= item['threshold']),
        'missing inputs': [c for c in FEATURE_COLUMNS if pd.isna(frame.loc[0, c])],
        'history entries': len(past), 'latest history date': str(past['date'].max().date()),
        'selection data end': str(selection_end.date())}

complete_mask = X_test_raw.notna().all(axis=1)
complete_index = modeling.loc[X_test_raw.index[complete_mask], ['raceId', 'driverId']].sort_values(['raceId', 'driverId']).index[0]
complete_record = X_test_raw.loc[complete_index].to_dict()
demo_date = modeling.loc[complete_index, 'date']
partial_record = complete_record.copy()
for field in ['grid_position', 'driver_prior5_podium_rate', 'constructor_standing_prior_position', 'circuitId']:
    partial_record[field] = np.nan
inference_complete = predict_pre_race(complete_record, demo_date)
inference_partial = predict_pre_race(partial_record, demo_date)
inference_table = pd.DataFrame([{'example': 'Complete', **inference_complete},
    {'example': 'Partially missing', **inference_partial}])
display(inference_table)
expected = float(predict_probability(fitted[best_name]['model'], X_test.loc[[complete_index]])[0])
assert np.isclose(inference_complete['podium probability'], expected, atol=1e-6)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(['Complete', 'Partially missing'], inference_table['podium probability'], color=['#287D8E', '#B95D4C'])
ax.axhline(inference_complete['threshold'], color='#334155', ls='--', label='Frozen validation threshold')
ax.set(ylabel='Predicted podium probability', ylim=(0, 1), title='Post-selection inference with complete and partial inputs')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_PATH}/7_inference_examples.png', dpi=150)
plt.show()


### Inference boundary regression checks

Reject malformed, contradictory and unavailable-at-selection-time inputs; preserve valid zero-grid/missing-grid cases. These checks validate the function, not predictive superiority.


In [ ]:
invalid_records = {
    'negative raw grid': {'grid_raw': -1}, 'negative ordinal grid': {'grid_position': -1},
    'fractional grid': {'grid_position': 1.5}, 'infinite grid': {'grid_raw': np.inf},
    'contradictory grid flag': {'grid_position': 3, 'grid_is_zero': 1},
    'inconsistent grid aliases': {'grid_raw': 2, 'grid_position': 3},
    'out of range form': {'driver_prior5_podium_rate': 3},
    'negative history count': {'driver_prior_entries': -1},
    'wrong calendar year': {'year': demo_date.year + 1}, 'forbidden outcome': {'positionOrder': 1},
    'invalid binary flag': {'grid_is_zero': 2}, 'nonscalar input': {'grid_position': [1, 2]},
    'one entry with impossible form': {'driver_prior_entries': 1, 'driver_prior5_podium_rate': 0.5},
    'two entries with impossible form': {'driver_prior_entries': 2, 'driver_prior5_podium_rate': 0.3},
    'five-entry window with impossible form': {'driver_prior_entries': 10, 'driver_prior5_podium_rate': 0.3},
    'unsupported raw grid': {'grid_raw': SUPPORTED_GRID_MAX + 1},
    'unsupported ordinal grid': {'grid_position': SUPPORTED_GRID_MAX + 1},
    'extreme finite grid': {'grid_raw': 999999}}
inference_check_rows = []
for case, record in invalid_records.items():
    try:
        predict_pre_race(record, demo_date)
    except ValueError as error:
        inference_check_rows.append({'case': case, 'passed': True, 'result': str(error)})
    else:
        raise AssertionError('Inference accepted invalid case: ' + case)
for case, record in {'known zero grid': {'grid_raw': 0}, 'missing grid': {},
                     'consistent aliases': {'grid_raw': 3, 'grid_position': 3, 'grid_is_zero': 0},
                     'one earlier nonpodium': {'driver_prior_entries': 1, 'driver_prior5_podium_rate': 0},
                     'one earlier podium': {'driver_prior_entries': 1, 'driver_prior5_podium_rate': 1},
                     'two-entry valid mean': {'driver_prior_entries': 2, 'driver_prior5_podium_rate': 0.5},
                     'five-entry valid mean': {'driver_prior_entries': 100, 'driver_prior5_podium_rate': 0.6},
                     'missing count preserves partial input': {'driver_prior5_podium_rate': 0.3},
                     'maximum supported grid': {'grid_raw': SUPPORTED_GRID_MAX},
                     'constructor fractional rate': {'constructor_prior_races': 1, 'constructor_prior5_podium_rate': 1/3}}.items():
    answer = predict_pre_race(record, demo_date)
    assert np.isfinite(answer['podium probability']) and 0 <= answer['podium probability'] <= 1
    inference_check_rows.append({'case': case, 'passed': True, 'result': 'Finite valid probability'})
validation_date = modeling.loc[X_val_raw.index, 'date'].min()
try:
    predict_pre_race({}, validation_date)
except ValueError:
    inference_check_rows.append({'case': 'selection-date guard', 'passed': True, 'result': 'Rejected before selection cutoff'})
else:
    raise AssertionError('Default inference accepted a date inside model selection.')
retrospective = predict_pre_race({}, validation_date, mode='retrospective')
assert retrospective['mode'] == 'retrospective'
inference_check_rows.append({'case': 'explicit retrospective demonstration', 'passed': True, 'result': 'Labeled retrospective'})
inference_checks = pd.DataFrame(inference_check_rows)
display(inference_checks)


## 8. Conclusions, limitations and same-run exports

Retrospective results define the recorded-entry population; the grid remains a
retrospective pre-race proxy. Unknown zero-grid reasons are not reconstructed
from later status/laps. Pre-race fields are resolved independently of outcomes,
and all earlier years are retained. Uncertain car/team assignments remain missing.
Constructor rebrands remain separate IDs; era and category drift limit transfer.

Row/feature removal, fill alternatives and the two required order experiments are
visible for all three model families. Feature-removal trials retrain; only Part 6
permutes values to measure frozen-model reliance. Explanations are not causal.

Validation is reused for tuning and a single seed is used. The test period was
viewed in earlier revisions: this remains a disclosed revised evaluation, not a
newly unseen holdout. No significance/calibration claim is made. Inference is
entry-level and does not constrain a race to exactly three positive predictions.

Export tables, figures, dataset hashes, dependencies, selected preparation state,
observed earlier input history and fitted models from this single execution. A
report built from these outputs must use the same run ID and values.


In [ ]:
preferred_rows = final_table[final_table['model'].eq(best_name)].set_index('split')
model_conclusion = (f'Preferred main model: {best_name}. Validation AP {preferred_rows.loc["validation", "PR-AUC"]:.4f}; '
    f'test AP {preferred_rows.loc["test", "PR-AUC"]:.4f}; test F1 {preferred_rows.loc["test", "F1"]:.4f}. '
    'Selection used validation only; test performance does not change the chosen model.')
display(Markdown(model_conclusion))

exports = {'primary_key_audit': pk_table, 'foreign_key_audit': fk_table,
    'q1_agreement_audit': q1_agreement_audit, 'shap_budget_audit': shap_budget_audit,
    'duplicate_input_audit': duplicate_input_audit,
    'q3_completion_conflicts': q3_completion_conflicts,
    'preprocessing_values': preprocessing_values, 'preprocessing_example': preprocessing_example,
    'cold_start_audit': cold_start_audit, 'final_nn_learning_summary': final_learning_summary,
    'inference_checks': inference_checks, 'model_attempts': attempt_table, 'model_comparison': final_table,
    'feature_ablations': ablation_table, 'preprocessing_experiments': order_table,
    'historical_fill_experiments': fill_table, 'combined_feature_experiments': feature_combination_table,
    'training_row_experiments': row_policy_table, 'training_row_populations': row_population_table,
    'retained_input_missingness': missing_input_audit, 'common_pipeline_selection': common_selection_table,
    'permutation_importance': permutation_table, 'shap_importance': shap_importance_table,
    'local_explanations': local_table, 'q1_rankings': ranking.reset_index(),
    'q1_bounds': grid_front_row_bounds.reset_index(), 'q2_controlled': q2_table.reset_index(),
    'q3_all_constructors': q3_table, 'q3_eligible': q3_eligible, 'q3_excluded': q3_excluded,
    'q3_matched_constructors': q3_matched,
    'inference_examples': inference_table, 'category_drift': drift_table.reset_index(),
    'preparation_audit': preparation_audit, 'split_summary': split_summary.reset_index()}
for label, table in exports.items():
    table.to_csv(OUTPUT_PATH / (label + '.csv'), index=False)
modeling.to_csv(OUTPUT_PATH / 'modeling_table.csv', index=False)
summary = {'versions': versions, 'modeling_rows': len(modeling), 'races': modeling['raceId'].nunique(),
    'features': X_train.shape[1], 'revision': PROJECT_REVISION,
    'test_already_viewed': True, 'selection_data_end': str(selection_end.date()),
    'duplicate_resolution': {'duplicate_pairs': int(len(duplicate_input_audit)),
        'ambiguous_grid_pairs': int(pre_race_entries['grid_conflict'].sum()),
        'ambiguous_constructor_pairs': int(pre_race_entries['constructor_conflict'].sum()),
        'post_race_fields_select_inputs': False},
    'q3_completion_conflicts': q3_completion_conflicts.to_dict('records'),
    'inference_checks_passed': int(inference_checks['passed'].sum()), 'selected_models': selected, 'preferred_model': best_name,
    'final_pipeline': final_pipeline, 'pipeline_conclusion': pipeline_conclusion,
    'q1_answer': q1_answer, 'q2_answer': q2_answer, 'q3_answer': q3_answer,
    'q3_matched_answer': q3_matched_answer,
    'model_conclusion': model_conclusion, 'experiment_conclusions': experiment_conclusions,
    'xai_conclusions': xai_conclusions, 'figures': sorted(p.name for p in FIG_PATH.glob('*.png'))}
(OUTPUT_PATH / 'results_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
assert selection_snapshot == (selected, best_name)
print('Exported', len(exports), 'tables, modeling table, figures and results_summary.json to', OUTPUT_PATH)
print('Completed all sections. All explanatory conclusions refer to computed results above.')


# Freeze source identity, dependency versions, preparation state and all final fitted models.
from datetime import datetime, timezone
import importlib.metadata
import platform
import sys
import joblib

def json_value(value):
    return value.item() if isinstance(value, np.generic) else str(value)

assert source_csv_hashes == {name + '.csv': hashlib.sha256(Path(DATA_PATH, name + '.csv').read_bytes()).hexdigest()
                            for name in TABLE_NAMES}
models_path = OUTPUT_PATH / 'models'
models_path.mkdir(exist_ok=True)
saved_models = {}
for number, (name, item) in enumerate(fitted.items(), start=1):
    path = models_path / (f'model_{number}.keras' if isinstance(item['model'], keras.Model) else f'model_{number}.joblib')
    if isinstance(item['model'], keras.Model):
        item['model'].save(path)
    else:
        joblib.dump(item['model'], path, compress=3)
    saved_models[name] = {'path': path.relative_to(OUTPUT_PATH).as_posix(), 'family': item['family'],
        'threshold': item['threshold'], 'settings': item['settings'], 'epochs_run': item['epochs'],
        'best_epoch': item['best_epoch'], 'input_columns': list(item['input_columns'])}
preprocessing_state = {'revision': PROJECT_REVISION, 'numeric_features': NUMERIC_FEATURES,
    'binary_features': BINARY_FEATURES, 'categorical_features': CATEGORICAL_FEATURES,
    'feature_columns': FEATURE_COLUMNS, 'category_levels': category_levels,
    'active_columns': list(ACTIVE_COLUMNS), 'fill_method': selected_fill,
    'cold_start': 'Standardized zero when no earlier observation exists, with no-history flag',
    'minimum_scale': 1e-8, 'training_data_end': str(training_end.date()),
    'selection_data_end': str(selection_end.date()), 'preferred_model': best_name,
    'inference_domain': {'grid_max': SUPPORTED_GRID_MAX,
        'grid_policy': 'Largest positive training grid; this is a supported API domain, not a universal sporting limit',
        'driver_form_policy': 'Mean of min(prior entries, 5) binary outcomes when both values are present',
        'driver_form_tolerance': 1e-6},
    'test_already_viewed': True, 'models': saved_models}
(OUTPUT_PATH / 'preprocessing_state.json').write_text(
    json.dumps(preprocessing_state, indent=2, default=json_value), encoding='utf-8')
modeling[['date'] + FEATURE_COLUMNS].to_csv(OUTPUT_PATH / 'feature_history.csv', index=False)
dependencies = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'tensorflow', 'keras',
                'shap', 'matplotlib', 'seaborn', 'missingno', 'IPython', 'joblib']
dependency_versions = {name: importlib.metadata.version(name) for name in dependencies}
(OUTPUT_PATH / 'requirements.txt').write_text(
    '\n'.join(name + '==' + version for name, version in dependency_versions.items()) + '\n', encoding='utf-8')
run_manifest = {'revision': PROJECT_REVISION, 'run_id': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version,
    'platform': platform.platform(), 'dependencies': dependency_versions,
    'data_sha256': source_csv_hashes, 'final_pipeline': final_pipeline,
    'test_already_viewed': True, 'selection_rule': 'Training/validation only; test does not select settings',
    'inference_checks_passed': int(inference_checks['passed'].sum()), 'saved_models': saved_models}
(OUTPUT_PATH / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2, default=json_value), encoding='utf-8')
summary['run_id'] = run_manifest['run_id']
summary['data_sha256'] = source_csv_hashes
summary['saved_models'] = saved_models
(OUTPUT_PATH / 'results_summary.json').write_text(json.dumps(summary, indent=2, default=json_value), encoding='utf-8')
print('Reproducibility bundle saved. Source CSV hashes unchanged. Inference checks:', len(inference_checks))
